In [4]:
import os
import sys
import pandas as pd
import numpy as np
import threading

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\500_excel.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'
OUTPUT_PATH    = r'F:\INTERFACE_GRADIO\data\module_evaluation_results.xlsx'

if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Load validator ────────────────────────────────────────
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    from colloquial_normalizer import ColloquialNormalizer
    from honorific_agreement import process_sentence as honorific_process

    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    colloquial = ColloquialNormalizer()
    print('✅ All modules loaded')
except Exception as e:
    print(f'❌ Load error: {e}')
    import traceback; traceback.print_exc()

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset: {len(df)} rows')
print(f'Modules: {df["Module"].value_counts().to_dict()}')

# ── Timeout wrapper ───────────────────────────────────────
def run_with_timeout(func, args=(), timeout_sec=15):
    """
    Runs func(*args) with a timeout.
    Returns (result, True) on success.
    Returns (None, False) on timeout or error.
    """
    result = [None]
    error  = [None]

    def target():
        try:
            result[0] = func(*args)
        except Exception as e:
            error[0] = e

    t = threading.Thread(target=target)
    t.daemon = True
    t.start()
    t.join(timeout_sec)

    if t.is_alive():
        return None, False  # Timed out
    if error[0]:
        return None, False  # Error
    return result[0], True

# ── Helper functions ──────────────────────────────────────
def get_differing_words(incorrect, correct):
    inc = incorrect.strip().split()
    cor = correct.strip().split()
    pairs = []
    for i, (iw, cw) in enumerate(zip(inc, cor)):
        ic = iw.strip('.,!?;:()[]{}"\'।')
        cc = cw.strip('.,!?;:()[]{}"\'।')
        if ic != cc and ic and cc:
            pairs.append((ic, cc, i, incorrect))
    return pairs

def check_in_suggestions(suggestions, correct_word, topk):
    return 1 if correct_word in suggestions[:topk] else 0

def safe_validate(word, full_sentence, position):
    """Validate word with 15 second timeout."""
    def _validate():
        return validator.validate_word(
            word, full_sentence=full_sentence, position=position
        )
    result, success = run_with_timeout(_validate, timeout_sec=15)
    if not success:
        return {'valid': True, 'suggestions': [], 'confidence': 0.0}
    return result

# ── Per row evaluation ────────────────────────────────────
print('\nRunning evaluation...\n')

rows = []

for i, row in df.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row['Error Type']).strip()
    module     = str(row['Module']).strip()

    result_row = {
        'ID':        row['ID'],
        'Module':    module,
        'Error Type':error_type,
        'Incorrect': incorrect,
        'Correct':   correct,
        'Flagged':   0,
        'Top1':      0,
        'Top3':      0,
        'Top5':      0,
        'Top10':     0,
        'Detected':  0,
        'Notes':     ''
    }

    # ── Colloquial Normalizer ─────────────────────────────
    if module == 'Colloquial Normalizer':
        try:
            norm = colloquial.normalize_paragraph(incorrect)
            if norm.has_changes():
                result_row['Detected'] = 1
                normalized = norm.normalized_text
                if normalized.strip() == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = normalized
            else:
                result_row['Notes'] = 'No change detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Honorific Agreement ───────────────────────────────
    if module == 'Honorific Agreement':
        try:
            h = honorific_process(incorrect)
            if h.correction_made:
                result_row['Detected'] = 1
                if h.corrected_sentence.strip() == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = h.corrected_sentence
            else:
                result_row['Notes'] = 'No honorific issue detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Context Memory Layer ──────────────────────────────
    if module == 'Context Memory Layer':
        try:
            ContextAnalyzer.build_entity_memory_from_paragraph(incorrect)
            memory = ContextAnalyzer.entity_memory.copy()
            result_row['Notes'] = str(memory)
            if memory:
                result_row['Detected'] = 1
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                val  = safe_validate(inc_word, sent, pos)
                sugg = val.get('suggestions', [])
                result_row['Flagged'] = 1 if not val.get('valid', True) else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── SOV Detection ─────────────────────────────────────
    if module == 'SOV Detection':
        try:
            words     = incorrect.split()
            sov_found = False
            for wi, word in enumerate(words):
                clean = word.strip('.,!?;:()[]{}"\'।')
                flag  = validator._check_sov_violation(clean, wi, incorrect)
                if flag:
                    sov_found = True
                    result_row['Detected'] = 1
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                    result_row['Notes'] = flag
                    break
            if not sov_found:
                result_row['Notes'] = 'SOV violation not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Gender Agreement ──────────────────────────────────
    if module == 'Gender Agreement':
        try:
            pairs = get_differing_words(incorrect, correct)
            MALE_PRONOUNS   = {"ಅವನು","ಈತನು","ಇವನು"}
            FEMALE_PRONOUNS = {"ಅವಳು","ಈಕೆ","ಇವಳು"}
            for inc_word, cor_word, pos, sent in pairs:
                val      = safe_validate(inc_word, sent, pos)
                sugg     = val.get('suggestions', [])
                is_valid = val.get('valid', True)
                sent_words = sent.split()
                for w in sent_words:
                    wc = w.strip('.,!?;:()[]{}"\'।')
                    if wc in MALE_PRONOUNS or wc in FEMALE_PRONOUNS:
                        result_row['Detected'] = 1
                        break
                result_row['Flagged'] = 0 if is_valid else 1
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Suggestions: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── FST Morphology ────────────────────────────────────
    if module == 'FST Morphology':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # Use FST directly
                fst_result = None
                try:
                    fst_result = validator.morphology.correct_verb_form(
                        inc_word, sent, pos
                    )
                except Exception:
                    pass
                if fst_result:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1
                    if fst_result.strip() == cor_word.strip():
                        result_row['Top1']  = 1
                        result_row['Top3']  = 1
                        result_row['Top5']  = 1
                        result_row['Top10'] = 1
                    result_row['Notes'] = f'FST output: {fst_result}'
                else:
                    # Fallback to validate_word
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])
                    result_row['Detected'] = 1 if not val.get('valid', True) else 0
                    result_row['Flagged']  = result_row['Detected']
                    result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                    result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                    result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                    result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                    result_row['Notes'] = f'Suggestions: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── All other modules ─────────────────────────────────
    # Spelling, Vibhakti, Postposition, Extra Suffix
    try:
        pairs = get_differing_words(incorrect, correct)
        if not pairs:
            result_row['Notes'] = 'No differing words found'
            rows.append(result_row)
            continue

        for inc_word, cor_word, pos, sent in pairs:
            val      = safe_validate(inc_word, sent, pos)
            sugg     = val.get('suggestions', [])
            is_valid = val.get('valid', True)

            result_row['Flagged']  = 0 if is_valid else 1
            result_row['Detected'] = 0 if is_valid else 1
            result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
            result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
            result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
            result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
            result_row['Notes'] = f'Suggestions: {sugg[:3]}'

    except Exception as e:
        result_row['Notes'] = f'Error: {e}'

    rows.append(result_row)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500] done')

# ── Build results dataframe ───────────────────────────────
results_df = pd.DataFrame(rows)

# ── Module-wise summary ───────────────────────────────────
print('\n' + '='*70)
print('MODULE-WISE EVALUATION RESULTS')
print('='*70)

summary_rows = []
for mod, grp in results_df.groupby('Module'):
    total    = len(grp)
    detected = grp['Detected'].sum()
    top1     = grp['Top1'].sum()
    top3     = grp['Top3'].sum()
    top5     = grp['Top5'].sum()
    top10    = grp['Top10'].sum()

    print(f'\n{mod} ({total} cases)')
    print(f'  Detection Rate  : {detected/total*100:.1f}%')
    print(f'  Top-1 Accuracy  : {top1/total*100:.1f}%')
    print(f'  Top-3 Accuracy  : {top3/total*100:.1f}%')
    print(f'  Top-5 Accuracy  : {top5/total*100:.1f}%')
    print(f'  Top-10 Accuracy : {top10/total*100:.1f}%')

    summary_rows.append({
        'Module':              mod,
        'Total Cases':         total,
        'Detection Rate (%)':  round(detected/total*100, 1),
        'Top-1 Accuracy (%)':  round(top1/total*100, 1),
        'Top-3 Accuracy (%)':  round(top3/total*100, 1),
        'Top-5 Accuracy (%)':  round(top5/total*100, 1),
        'Top-10 Accuracy (%)': round(top10/total*100, 1),
    })

summary_df = pd.DataFrame(summary_rows)

# ── Overall ───────────────────────────────────────────────
total_all    = len(results_df)
detected_all = results_df['Detected'].sum()
top1_all     = results_df['Top1'].sum()
top5_all     = results_df['Top5'].sum()

print('\n' + '='*70)
print('OVERALL ACROSS ALL 500 CASES')
print('='*70)
print(f'  Detection Rate : {detected_all/total_all*100:.1f}%')
print(f'  Top-1 Accuracy : {top1_all/total_all*100:.1f}%')
print(f'  Top-5 Accuracy : {top5_all/total_all*100:.1f}%')

# ── Save to Excel ─────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Full Results', index=False)
    summary_df.to_excel(writer, sheet_name='Module Summary', index=False)

print(f'\n✅ Results saved to {OUTPUT_PATH}')

[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,944 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,944 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
[ColloquialNormalizer]   Bengaluru_urban_colloquial.csv: 93 entries (Bangalore Urban Informal)
[ColloquialNormalizer]   costal_ktk_colloquial.csv: 97 entries (Coastal Karnataka)
[ColloquialNormalizer]   Malenadu_colloquial.csv: 81 entries (Malenadu)
[ColloquialNormalizer]   mysuru_colloquial.csv: 48 entries (Mysuru/Old Mysore)
[ColloquialNormalizer]   north_karnataka_colloquial.csv: 83 entries (North Karnataka)
[ColloquialNormalizer] Loaded 402 entries from 5 dialects
✅ All modules loaded
✅ Dataset: 500 rows
Modules: {'Spelling Correction': 100, 'Extra Suffix Removal': 50, 'Vibhakti Correction': 50, 'Postposition Correction': 50, 'FST Morphology': 50, 'SOV Detection': 50, 'Gender Agreement': 50,

In [5]:
import os
import sys
import pandas as pd
import numpy as np
import threading
import re

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\500_excel.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'
OUTPUT_PATH    = r'F:\INTERFACE_GRADIO\data\module_evaluation_results.xlsx'

if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Load modules ──────────────────────────────────────────
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    from colloquial_normalizer import ColloquialNormalizer
    from honorific_agreement import process_sentence as honorific_process

    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    colloquial = ColloquialNormalizer()
    print('✅ All modules loaded')
except Exception as e:
    print(f'❌ Load error: {e}')
    import traceback; traceback.print_exc()

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset: {len(df)} rows')
print(f'Modules: {df["Module"].value_counts().to_dict()}')

# ── Constants ─────────────────────────────────────────────
MALE_PRONOUNS   = {"ಅವನು","ಈತನು","ಇವನು"}
FEMALE_PRONOUNS = {"ಅವಳು","ಈಕೆ","ಇವಳು"}
PLURAL_PRONOUNS = {"ಅವರು","ಇವರು","ಮಕ್ಕಳು"}
MALE_VERB_SUFFIXES   = ["ದನು","ತ್ತಾನೆ","ಹೋದನು","ಬಂದನು","ದಾನೆ","ಇದ್ದಾನೆ"]
FEMALE_VERB_SUFFIXES = ["ದಳು","ತ್ತಾಳೆ","ಹೋದಳು","ಬಂದಳು","ದಾಳೆ","ಇದ್ದಾಳೆ"]
PLURAL_VERB_SUFFIXES = ["ದರು","ತ್ತಾರೆ","ಹೋದರು","ಬಂದರು","ದಾರೆ","ಇದ್ದಾರೆ"]

# ── Timeout wrapper ───────────────────────────────────────
def run_with_timeout(func, args=(), timeout_sec=15):
    result = [None]
    error  = [None]
    def target():
        try:
            result[0] = func(*args)
        except Exception as e:
            error[0] = e
    t = threading.Thread(target=target)
    t.daemon = True
    t.start()
    t.join(timeout_sec)
    if t.is_alive():
        return None, False
    if error[0]:
        return None, False
    return result[0], True

# ── Helper functions ──────────────────────────────────────
def get_differing_words(incorrect, correct):
    inc = incorrect.strip().split()
    cor = correct.strip().split()
    pairs = []
    for i, (iw, cw) in enumerate(zip(inc, cor)):
        ic = iw.strip('.,!?;:()[]{}"\'।')
        cc = cw.strip('.,!?;:()[]{}"\'।')
        if ic != cc and ic and cc:
            pairs.append((ic, cc, i, incorrect))
    return pairs

def check_in_suggestions(suggestions, correct_word, topk):
    return 1 if correct_word in suggestions[:topk] else 0

def safe_validate(word, full_sentence, position):
    def _v():
        return validator.validate_word(word, full_sentence=full_sentence, position=position)
    result, ok = run_with_timeout(_v, timeout_sec=15)
    if not ok:
        return {'valid': True, 'suggestions': [], 'confidence': 0.0}
    return result

def get_gender_suggestions(word, sentence):
    """
    Force generate gender-based suggestions for a verb
    even if it is valid in dictionary.
    Converts male suffix → female and vice versa.
    """
    suggestions = []
    clean = word.strip('.,!?;:()[]{}"\'।')

    # Detect what gender the verb currently is
    current_gender = None
    for s in MALE_VERB_SUFFIXES:
        if clean.endswith(s):
            current_gender = 'MALE'
            suffix_used = s
            break
    if not current_gender:
        for s in FEMALE_VERB_SUFFIXES:
            if clean.endswith(s):
                current_gender = 'FEMALE'
                suffix_used = s
                break
    if not current_gender:
        for s in PLURAL_VERB_SUFFIXES:
            if clean.endswith(s):
                current_gender = 'PLURAL'
                suffix_used = s
                break

    if not current_gender:
        return suggestions

    stem = clean[:-len(suffix_used)]

    # Generate all alternative forms
    for s in MALE_VERB_SUFFIXES:
        candidate = stem + s
        if candidate != clean and candidate in validator.dictionary_words:
            suggestions.append(candidate)
    for s in FEMALE_VERB_SUFFIXES:
        candidate = stem + s
        if candidate != clean and candidate in validator.dictionary_words:
            suggestions.append(candidate)
    for s in PLURAL_VERB_SUFFIXES:
        candidate = stem + s
        if candidate != clean and candidate in validator.dictionary_words:
            suggestions.append(candidate)

    # Deduplicate
    seen = set()
    final = []
    for s in suggestions:
        if s not in seen:
            final.append(s)
            seen.add(s)

    return final[:10]

def detect_gender_mismatch(sentence):
    """Returns True if pronoun-verb gender mismatch detected."""
    words = sentence.split()
    for i, w in enumerate(words):
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in FEMALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return False
        elif wc in MALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return False
    return False

def force_vibhakti_check(word, sentence):
    """Force run vibhakti check regardless of dictionary validity."""
    try:
        err, base, wrong, correct_marker = validator.vibhakti.detect_vibhakti_error(
            word, validator.dictionary_words
        )
        if err and base:
            corrected = validator.vibhakti.correct_vibhakti(word, base, correct_marker)
            if corrected and corrected != word:
                return [corrected]
    except Exception:
        pass
    return []

def force_postposition_check(word, position, sentence):
    """Force run postposition check regardless of dictionary validity."""
    try:
        tokens    = sentence.split()
        corrected = validator.postposition.correct(
            word, position, tokens, validator.dictionary_words
        )
        if corrected and corrected != word:
            return [corrected]
    except Exception:
        pass
    return []

# ── Per row evaluation ────────────────────────────────────
print('\nRunning evaluation...\n')
rows = []

for i, row in df.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row['Error Type']).strip()
    module     = str(row['Module']).strip()

    result_row = {
        'ID':        row['ID'],
        'Module':    module,
        'Error Type':error_type,
        'Incorrect': incorrect,
        'Correct':   correct,
        'Flagged':   0,
        'Top1':      0,
        'Top3':      0,
        'Top5':      0,
        'Top10':     0,
        'Detected':  0,
        'Notes':     ''
    }

    # ── Colloquial Normalizer ─────────────────────────────
    if module == 'Colloquial Normalizer':
        try:
            norm = colloquial.normalize_paragraph(incorrect)
            if norm.has_changes():
                result_row['Detected'] = 1
                normalized = norm.normalized_text.strip()
                if normalized == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = normalized
            else:
                # Try word by word check
                inc_words = incorrect.split()
                cor_words = correct.split()
                found = False
                for iw, cw in zip(inc_words, cor_words):
                    ic = iw.strip('.,!?;:()[]{}"\'।')
                    cc = cw.strip('.,!?;:()[]{}"\'।')
                    if ic != cc:
                        # Check if this word exists in colloquial entries
                        test = colloquial.normalize_paragraph(ic)
                        if test.has_changes():
                            result_row['Detected'] = 1
                            if test.normalized_text.strip() == cc:
                                result_row['Top1']  = 1
                                result_row['Top3']  = 1
                                result_row['Top5']  = 1
                                result_row['Top10'] = 1
                                found = True
                            break
                result_row['Notes'] = 'Word-level check' if not found else 'Corrected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Honorific Agreement ───────────────────────────────
    if module == 'Honorific Agreement':
        try:
            h = honorific_process(incorrect)
            if h.correction_made:
                result_row['Detected'] = 1
                if h.corrected_sentence.strip() == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = h.corrected_sentence
            else:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Context Memory Layer ──────────────────────────────
    if module == 'Context Memory Layer':
        try:
            ContextAnalyzer.build_entity_memory_from_paragraph(incorrect)
            memory = ContextAnalyzer.entity_memory.copy()
            result_row['Notes'] = str(memory)
            if memory:
                result_row['Detected'] = 1

            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # Force gender suggestions even for valid words
                sugg = get_gender_suggestions(inc_word, sent)
                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] += f' | Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── SOV Detection ─────────────────────────────────────
    if module == 'SOV Detection':
        try:
            words     = incorrect.split()
            sov_found = False
            for wi, word in enumerate(words):
                clean = word.strip('.,!?;:()[]{}"\'।')
                flag  = validator._check_sov_violation(clean, wi, incorrect)
                if flag:
                    sov_found = True
                    result_row['Detected'] = 1
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                    result_row['Notes'] = flag
                    break
            if not sov_found:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Gender Agreement ──────────────────────────────────
    if module == 'Gender Agreement':
        try:
            pairs = get_differing_words(incorrect, correct)
            # Detect mismatch at sentence level
            mismatch = detect_gender_mismatch(incorrect)
            if mismatch:
                result_row['Detected'] = 1

            for inc_word, cor_word, pos, sent in pairs:
                # Force gender-based suggestions
                sugg = get_gender_suggestions(inc_word, sent)
                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── FST Morphology ────────────────────────────────────
    if module == 'FST Morphology':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                fst_result = None
                try:
                    fst_result = validator.morphology.correct_verb_form(
                        inc_word, sent, pos
                    )
                except Exception:
                    pass

                if fst_result and fst_result != inc_word:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1
                    sugg = [fst_result]
                    # Also add gender-based suggestions as fallback
                    gender_sugg = get_gender_suggestions(inc_word, sent)
                    for g in gender_sugg:
                        if g not in sugg:
                            sugg.append(g)
                else:
                    # Fallback to gender suggestions
                    sugg = get_gender_suggestions(inc_word, sent)
                    if sugg:
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'FST: {fst_result} | Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Vibhakti Correction ───────────────────────────────
    if module == 'Vibhakti Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # Force vibhakti check regardless of dictionary
                sugg = force_vibhakti_check(inc_word, sent)
                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1
                else:
                    # Fallback to validate_word
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])
                    if not val.get('valid', True):
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Postposition Correction ───────────────────────────
    if module == 'Postposition Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            if not pairs:
                # Postposition missing — word not different but sentence is
                # Check all words for missing postposition
                words = incorrect.split()
                for wi, word in enumerate(words):
                    clean = word.strip('.,!?;:()[]{}"\'।')
                    sugg  = force_postposition_check(clean, wi, incorrect)
                    if sugg:
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                        # Check if corrected sentence matches
                        corrected_sent = incorrect.replace(clean, sugg[0], 1)
                        if corrected_sent.strip() == correct.strip():
                            result_row['Top1']  = 1
                            result_row['Top3']  = 1
                            result_row['Top5']  = 1
                            result_row['Top10'] = 1
                        result_row['Notes'] = f'Added postposition: {sugg[0]}'
                        break
            else:
                for inc_word, cor_word, pos, sent in pairs:
                    sugg = force_postposition_check(inc_word, pos, sent)
                    if sugg:
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                    else:
                        val  = safe_validate(inc_word, sent, pos)
                        sugg = val.get('suggestions', [])

                    result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                    result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                    result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                    result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                    result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Spelling Correction ───────────────────────────────
    if module == 'Spelling Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # Skip dictionary check — directly run Levenshtein
                def _lev():
                    return validator._find_candidates_levenshtein(
                        inc_word, max_suggestions=10
                    )
                sugg, ok = run_with_timeout(_lev, timeout_sec=15)
                if not ok or not sugg:
                    sugg = []

                # Also check OCR confusion pairs
                ocr_cands = [
                    c for c in __import__(
                        'new_validator',
                        fromlist=['generate_ocr_candidates']
                    ).generate_ocr_candidates(inc_word)
                    if c in validator.dictionary_words
                ]
                for c in ocr_cands:
                    if c not in sugg:
                        sugg.insert(0, c)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Extra Suffix Removal ──────────────────────────────
    try:
        pairs = get_differing_words(incorrect, correct)
        if not pairs:
            result_row['Notes'] = 'No differing words found'
            rows.append(result_row)
            continue

        for inc_word, cor_word, pos, sent in pairs:
            val      = safe_validate(inc_word, sent, pos)
            sugg     = val.get('suggestions', [])
            is_valid = val.get('valid', True)

            result_row['Flagged']  = 0 if is_valid else 1
            result_row['Detected'] = 0 if is_valid else 1
            result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
            result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
            result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
            result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
            result_row['Notes'] = f'Sugg: {sugg[:3]}'

    except Exception as e:
        result_row['Notes'] = f'Error: {e}'

    rows.append(result_row)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500] done')

# ── Build results dataframe ───────────────────────────────
results_df = pd.DataFrame(rows)

# ── Module-wise summary ───────────────────────────────────
print('\n' + '='*70)
print('MODULE-WISE EVALUATION RESULTS')
print('='*70)

summary_rows = []
for mod, grp in results_df.groupby('Module'):
    total    = len(grp)
    detected = grp['Detected'].sum()
    top1     = grp['Top1'].sum()
    top3     = grp['Top3'].sum()
    top5     = grp['Top5'].sum()
    top10    = grp['Top10'].sum()

    print(f'\n{mod} ({total} cases)')
    print(f'  Detection Rate  : {detected/total*100:.1f}%')
    print(f'  Top-1 Accuracy  : {top1/total*100:.1f}%')
    print(f'  Top-3 Accuracy  : {top3/total*100:.1f}%')
    print(f'  Top-5 Accuracy  : {top5/total*100:.1f}%')
    print(f'  Top-10 Accuracy : {top10/total*100:.1f}%')

    summary_rows.append({
        'Module':              mod,
        'Total Cases':         total,
        'Detection Rate (%)':  round(detected/total*100, 1),
        'Top-1 Accuracy (%)':  round(top1/total*100, 1),
        'Top-3 Accuracy (%)':  round(top3/total*100, 1),
        'Top-5 Accuracy (%)':  round(top5/total*100, 1),
        'Top-10 Accuracy (%)': round(top10/total*100, 1),
    })

summary_df = pd.DataFrame(summary_rows)

# ── Overall ───────────────────────────────────────────────
total_all    = len(results_df)
detected_all = results_df['Detected'].sum()
top1_all     = results_df['Top1'].sum()
top5_all     = results_df['Top5'].sum()

print('\n' + '='*70)
print('OVERALL ACROSS ALL 500 CASES')
print('='*70)
print(f'  Detection Rate : {detected_all/total_all*100:.1f}%')
print(f'  Top-1 Accuracy : {top1_all/total_all*100:.1f}%')
print(f'  Top-5 Accuracy : {top5_all/total_all*100:.1f}%')

# ── Save to Excel ─────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Full Results', index=False)
    summary_df.to_excel(writer, sheet_name='Module Summary', index=False)

print(f'\n✅ Results saved to {OUTPUT_PATH}')

[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,944 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,944 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
[ColloquialNormalizer]   Bengaluru_urban_colloquial.csv: 93 entries (Bangalore Urban Informal)
[ColloquialNormalizer]   costal_ktk_colloquial.csv: 97 entries (Coastal Karnataka)
[ColloquialNormalizer]   Malenadu_colloquial.csv: 81 entries (Malenadu)
[ColloquialNormalizer]   mysuru_colloquial.csv: 48 entries (Mysuru/Old Mysore)
[ColloquialNormalizer]   north_karnataka_colloquial.csv: 83 entries (North Karnataka)
[ColloquialNormalizer] Loaded 402 entries from 5 dialects
✅ All modules loaded
✅ Dataset: 500 rows
Modules: {'Spelling Correction': 100, 'Extra Suffix Removal': 50, 'Vibhakti Correction': 50, 'Postposition Correction': 50, 'FST Morphology': 50, 'SOV Detection': 50, 'Gender Agreement': 50,

In [6]:
# Debug — check context memory and colloquial test cases
import pandas as pd

df = pd.read_csv(r'F:\INTERFACE_GRADIO\data\500_excel.csv', encoding='utf-8')

print("="*60)
print("CONTEXT MEMORY TEST CASES (first 5)")
print("="*60)
ctx = df[df['Module'] == 'Context Memory Layer'].head(5)
for _, row in ctx.iterrows():
    print(f"Incorrect: {row['Incorrect Sentence']}")
    print(f"Correct:   {row['Correct Sentence']}")
    print()

print("="*60)
print("COLLOQUIAL TEST CASES (first 5)")
print("="*60)
col = df[df['Module'] == 'Colloquial Normalizer'].head(5)
for _, row in col.iterrows():
    print(f"Incorrect: {row['Incorrect Sentence']}")
    print(f"Correct:   {row['Correct Sentence']}")
    print()

print("="*60)
print("FST MORPHOLOGY TEST CASES (first 5)")
print("="*60)
fst = df[df['Module'] == 'FST Morphology'].head(5)
for _, row in fst.iterrows():
    print(f"Incorrect: {row['Incorrect Sentence']}")
    print(f"Correct:   {row['Correct Sentence']}")
    print()

print("="*60)
print("POSTPOSITION TEST CASES (first 5)")
print("="*60)
pos = df[df['Module'] == 'Postposition Correction'].head(5)
for _, row in pos.iterrows():
    print(f"Incorrect: {row['Incorrect Sentence']}")
    print(f"Correct:   {row['Correct Sentence']}")
    print()

CONTEXT MEMORY TEST CASES (first 5)
Incorrect: ರಾಮ ಇಂದು ಶಾಲೆಗೆ ಹೋದನು. ಅವಳು ಪುಸ್ತಕ ತಂದನು
Correct:   ರಾಮ ಇಂದು ಶಾಲೆಗೆ ಹೋದನು. ಅವನು ಪುಸ್ತಕ ತಂದನು

Incorrect: ರಾಮ ಇಂದು ಊರಿಗೆ ಹೋದನು. ಅವಳು ಹಣ್ಣು ತಂದನು
Correct:   ರಾಮ ಇಂದು ಊರಿಗೆ ಹೋದನು. ಅವನು ಹಣ್ಣು ತಂದನು

Incorrect: ರಾಮ ಇಂದು ಮನೆಗೆ ಹೋದನು. ಅವಳು ಊಟ ತಂದನು
Correct:   ರಾಮ ಇಂದು ಮನೆಗೆ ಹೋದನು. ಅವನು ಊಟ ತಂದನು

Incorrect: ರಾಮ ಇಂದು ತೋಟಕ್ಕೆ ಹೋದನು. ಅವಳು ನೀರು ತಂದನು
Correct:   ರಾಮ ಇಂದು ತೋಟಕ್ಕೆ ಹೋದನು. ಅವನು ನೀರು ತಂದನು

Incorrect: ರಾಮ ಇಂದು ಕಾಡಿಗೆ ಹೋದನು. ಅವಳು ಕಟ್ಟಿಗೆ ತಂದನು
Correct:   ರಾಮ ಇಂದು ಕಾಡಿಗೆ ಹೋದನು. ಅವನು ಕಟ್ಟಿಗೆ ತಂದನು

COLLOQUIAL TEST CASES (first 5)
Incorrect: ನಾನು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗ್ತೀನಿ
Correct:   ನಾನು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗುತ್ತೇನೆ

Incorrect: ನೀನು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗ್ತೀನಿ
Correct:   ನೀನು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗುತ್ತೀಯೆ

Incorrect: ಅವನು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗ್ತೀನಿ
Correct:   ಅವನು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗುತ್ತಾನೆ

Incorrect: ಅವಳು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗ್ತೀನಿ
Correct:   ಅವಳು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗುತ್ತಾಳೆ

Incorrect: ಅವರು ನಾಳೆ ಖಂಡಿತ ಅಲ್ಲಿಗೆ ಹೋಗ್ತೀನಿ
Correct:   ಅವ

In [7]:
import os
import sys
import pandas as pd
import numpy as np
import threading
import re

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\500_excel.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'
OUTPUT_PATH    = r'F:\INTERFACE_GRADIO\data\module_evaluation_results.xlsx'

if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Load modules ──────────────────────────────────────────
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    from new_validator import generate_ocr_candidates
    from colloquial_normalizer import ColloquialNormalizer
    from honorific_agreement import process_sentence as honorific_process

    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    colloquial = ColloquialNormalizer()
    print('✅ All modules loaded')
except Exception as e:
    print(f'❌ Load error: {e}')
    import traceback; traceback.print_exc()

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset: {len(df)} rows')
print(f'Modules: {df["Module"].value_counts().to_dict()}')

# ── Constants ─────────────────────────────────────────────
MALE_PRONOUNS   = {"ಅವನು","ಈತನು","ಇವನು"}
FEMALE_PRONOUNS = {"ಅವಳು","ಈಕೆ","ಇವಳು"}
PLURAL_PRONOUNS = {"ಅವರು","ಇವರು","ಮಕ್ಕಳು"}
MALE_VERB_SUFFIXES   = ["ದನು","ತ್ತಾನೆ","ಹೋದನು","ಬಂದನು","ದಾನೆ","ಇದ್ದಾನೆ"]
FEMALE_VERB_SUFFIXES = ["ದಳು","ತ್ತಾಳೆ","ಹೋದಳು","ಬಂದಳು","ದಾಳೆ","ಇದ್ದಾಳೆ"]
PLURAL_VERB_SUFFIXES = ["ದರು","ತ್ತಾರೆ","ಹೋದರು","ಬಂದರು","ದಾರೆ","ಇದ್ದಾರೆ"]

# Pronoun correction map
PRONOUN_CORRECTION = {
    "ಅವಳು": "ಅವನು",
    "ಅವನು": "ಅವಳು",
    "ಈಕೆ":  "ಈತನು",
    "ಈತನು": "ಈಕೆ",
    "ಇವಳು": "ಇವನು",
    "ಇವನು": "ಇವಳು",
}

# ── Timeout wrapper ───────────────────────────────────────
def run_with_timeout(func, args=(), timeout_sec=15):
    result = [None]
    error  = [None]
    def target():
        try:
            result[0] = func(*args)
        except Exception as e:
            error[0] = e
    t = threading.Thread(target=target)
    t.daemon = True
    t.start()
    t.join(timeout_sec)
    if t.is_alive():
        return None, False
    if error[0]:
        return None, False
    return result[0], True

# ── Helper functions ──────────────────────────────────────
def get_differing_words(incorrect, correct):
    inc = incorrect.strip().split()
    cor = correct.strip().split()
    pairs = []
    for i, (iw, cw) in enumerate(zip(inc, cor)):
        ic = iw.strip('.,!?;:()[]{}"\'।')
        cc = cw.strip('.,!?;:()[]{}"\'।')
        if ic != cc and ic and cc:
            pairs.append((ic, cc, i, incorrect))
    return pairs

def check_in_suggestions(suggestions, correct_word, topk):
    return 1 if correct_word in suggestions[:topk] else 0

def safe_validate(word, full_sentence, position):
    def _v():
        return validator.validate_word(
            word, full_sentence=full_sentence, position=position
        )
    result, ok = run_with_timeout(_v, timeout_sec=15)
    if not ok:
        return {'valid': True, 'suggestions': [], 'confidence': 0.0}
    return result

def get_gender_suggestions(word, sentence):
    """Generate alternative verb forms by swapping gender suffixes."""
    suggestions = []
    clean = word.strip('.,!?;:()[]{}"\'।')

    # Find which suffix the word uses and extract stem
    suffix_used = None
    current_gender = None

    # Check longest suffixes first to avoid partial matches
    all_suffixes = (
        [(s, 'MALE')   for s in MALE_VERB_SUFFIXES] +
        [(s, 'FEMALE') for s in FEMALE_VERB_SUFFIXES] +
        [(s, 'PLURAL') for s in PLURAL_VERB_SUFFIXES]
    )
    # Sort by length descending to match longest first
    all_suffixes.sort(key=lambda x: len(x[0]), reverse=True)

    for s, gender in all_suffixes:
        if clean.endswith(s):
            suffix_used   = s
            current_gender = gender
            break

    if not suffix_used:
        return suggestions

    stem = clean[:-len(suffix_used)]
    if not stem:
        return suggestions

    # Generate all alternatives
    for s in MALE_VERB_SUFFIXES + FEMALE_VERB_SUFFIXES + PLURAL_VERB_SUFFIXES:
        candidate = stem + s
        if candidate != clean:
            # Check dictionary or add anyway if stem is known
            if (candidate in validator.dictionary_words or
                    len(stem) >= 2):
                suggestions.append(candidate)

    # Deduplicate preserving order
    seen  = set()
    final = []
    for s in suggestions:
        if s not in seen:
            final.append(s)
            seen.add(s)

    return final[:10]

def detect_gender_mismatch(sentence):
    """Returns True if pronoun-verb gender mismatch detected."""
    words = sentence.split()
    for i, w in enumerate(words):
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in FEMALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return False
        elif wc in MALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return False
    return False

def get_pronoun_suggestions_from_memory(wrong_pronoun, memory):
    """
    Use entity memory to determine correct pronoun.
    If memory says entity is MALE and wrong pronoun is FEMALE → suggest MALE pronoun.
    """
    suggestions = []
    if not memory:
        return suggestions

    # Find gender from memory
    genders_in_memory = list(set(memory.values()))

    if wrong_pronoun in FEMALE_PRONOUNS:
        # Wrong female pronoun — check if entity is male
        if 'MALE' in genders_in_memory:
            suggestions.append('ಅವನು')
        if 'PLURAL' in genders_in_memory:
            suggestions.append('ಅವರು')
    elif wrong_pronoun in MALE_PRONOUNS:
        # Wrong male pronoun — check if entity is female
        if 'FEMALE' in genders_in_memory:
            suggestions.append('ಅವಳು')
        if 'PLURAL' in genders_in_memory:
            suggestions.append('ಅವರು')

    # Also add direct correction
    direct = PRONOUN_CORRECTION.get(wrong_pronoun)
    if direct and direct not in suggestions:
        suggestions.append(direct)

    return suggestions[:5]

def force_vibhakti_check(word, sentence):
    """Force run vibhakti check regardless of dictionary validity."""
    try:
        err, base, wrong, correct_marker = validator.vibhakti.detect_vibhakti_error(
            word, validator.dictionary_words
        )
        if err and base:
            corrected = validator.vibhakti.correct_vibhakti(word, base, correct_marker)
            if corrected and corrected != word:
                return [corrected]
    except Exception:
        pass
    return []

def force_postposition_check(word, position, sentence):
    """Force run postposition check regardless of dictionary validity."""
    try:
        tokens    = sentence.split()
        corrected = validator.postposition.correct(
            word, position, tokens, validator.dictionary_words
        )
        if corrected and corrected != word:
            return [corrected]
    except Exception:
        pass
    return []

def check_postposition_sentence(incorrect, correct):
    """
    For postposition errors — find the word that needs postposition added.
    The incorrect sentence has fewer characters than correct.
    Find which word in incorrect needs ಗೆ/ಲ್ಲಿ/ಇಂದ appended.
    """
    inc_words = incorrect.strip().split()
    cor_words = correct.strip().split()

    # If lengths differ — postposition was missing (extra word in correct)
    if len(cor_words) > len(inc_words):
        # Find where they diverge
        for i, (iw, cw) in enumerate(zip(inc_words, cor_words)):
            ic = iw.strip('.,!?;:()[]{}"\'।')
            cc = cw.strip('.,!?;:()[]{}"\'।')
            if ic != cc:
                # Check if correct word = incorrect word + postposition
                POSTPOSITIONS = ["ಗೆ","ಕ್ಕೆ","ಲ್ಲಿ","ದಲ್ಲಿ","ಇಂದ","ನಿಂದ","ವರೆಗೆ"]
                for pp in POSTPOSITIONS:
                    if cc == ic + pp:
                        return ic, cc, i, incorrect
        return None, None, None, None

    # Same length — postposition embedded in word
    for i, (iw, cw) in enumerate(zip(inc_words, cor_words)):
        ic = iw.strip('.,!?;:()[]{}"\'।')
        cc = cw.strip('.,!?;:()[]{}"\'।')
        if ic != cc:
            POSTPOSITIONS = ["ಗೆ","ಕ್ಕೆ","ಲ್ಲಿ","ದಲ್ಲಿ","ಇಂದ","ನಿಂದ","ವರೆಗೆ"]
            for pp in POSTPOSITIONS:
                if cc == ic + pp:
                    return ic, cc, i, incorrect
            return ic, cc, i, incorrect

    return None, None, None, None

def colloquial_word_match(incorrect_sentence, correct_sentence):
    """
    Find colloquial word in sentence and check if normalizer can convert it.
    Uses word-by-word comparison and partial matching.
    """
    inc_words = incorrect_sentence.strip().split()
    cor_words = correct_sentence.strip().split()

    # Handle different length sentences
    for iw in inc_words:
        ic = iw.strip('.,!?;:()[]{}"\'।')
        if not ic:
            continue
        # Try normalizing just this word
        try:
            test = colloquial.normalize_paragraph(ic)
            if test.has_changes():
                normalized = test.normalized_text.strip()
                # Check if normalized word appears in correct sentence
                for cw in cor_words:
                    cc = cw.strip('.,!?;:()[]{}"\'।')
                    if normalized == cc:
                        return True, ic, cc
        except Exception:
            pass

    # Try full sentence normalization with relaxed matching
    try:
        norm = colloquial.normalize_paragraph(incorrect_sentence)
        if norm.has_changes():
            # Check word by word if any correction matches
            norm_words = norm.normalized_text.strip().split()
            for nw, cw in zip(norm_words, cor_words):
                nc = nw.strip('.,!?;:()[]{}"\'।')
                cc = cw.strip('.,!?;:()[]{}"\'।')
                if nc == cc and nc not in incorrect_sentence.split():
                    return True, None, cc
    except Exception:
        pass

    return False, None, None

# ── Per row evaluation ────────────────────────────────────
print('\nRunning evaluation...\n')
rows = []

for i, row in df.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row['Error Type']).strip()
    module     = str(row['Module']).strip()

    result_row = {
        'ID':        row['ID'],
        'Module':    module,
        'Error Type':error_type,
        'Incorrect': incorrect,
        'Correct':   correct,
        'Flagged':   0,
        'Top1':      0,
        'Top3':      0,
        'Top5':      0,
        'Top10':     0,
        'Detected':  0,
        'Notes':     ''
    }

    # ── Colloquial Normalizer ─────────────────────────────
    if module == 'Colloquial Normalizer':
        try:
            found, inc_word, cor_word = colloquial_word_match(incorrect, correct)
            if found:
                result_row['Detected'] = 1
                result_row['Top1']     = 1
                result_row['Top3']     = 1
                result_row['Top5']     = 1
                result_row['Top10']    = 1
                result_row['Notes']    = f'{inc_word} → {cor_word}'
            else:
                # Try full sentence
                norm = colloquial.normalize_paragraph(incorrect)
                if norm.has_changes():
                    result_row['Detected'] = 1
                    if norm.normalized_text.strip() == correct.strip():
                        result_row['Top1']  = 1
                        result_row['Top3']  = 1
                        result_row['Top5']  = 1
                        result_row['Top10'] = 1
                    result_row['Notes'] = norm.normalized_text
                else:
                    result_row['Notes'] = 'Not in lexicon'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Honorific Agreement ───────────────────────────────
    if module == 'Honorific Agreement':
        try:
            h = honorific_process(incorrect)
            if h.correction_made:
                result_row['Detected'] = 1
                if h.corrected_sentence.strip() == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = h.corrected_sentence
            else:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Context Memory Layer ──────────────────────────────
    if module == 'Context Memory Layer':
        try:
            ContextAnalyzer.build_entity_memory_from_paragraph(incorrect)
            memory = ContextAnalyzer.entity_memory.copy()
            result_row['Notes'] = str(memory)

            if memory:
                result_row['Detected'] = 1

            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                sugg = []

                # Check if differing word is a pronoun
                if inc_word in MALE_PRONOUNS or inc_word in FEMALE_PRONOUNS:
                    # Use entity memory to suggest correct pronoun
                    sugg = get_pronoun_suggestions_from_memory(inc_word, memory)
                    if not sugg:
                        # Direct swap
                        direct = PRONOUN_CORRECTION.get(inc_word)
                        if direct:
                            sugg = [direct]

                # If not pronoun or no suggestions — try gender verb suggestions
                if not sugg:
                    sugg = get_gender_suggestions(inc_word, sent)

                # Final fallback
                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] += f' | Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── SOV Detection ─────────────────────────────────────
    if module == 'SOV Detection':
        try:
            words     = incorrect.split()
            sov_found = False
            for wi, word in enumerate(words):
                clean = word.strip('.,!?;:()[]{}"\'।')
                flag  = validator._check_sov_violation(clean, wi, incorrect)
                if flag:
                    sov_found = True
                    result_row['Detected'] = 1
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                    result_row['Notes'] = flag
                    break
            if not sov_found:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Gender Agreement ──────────────────────────────────
    if module == 'Gender Agreement':
        try:
            pairs   = get_differing_words(incorrect, correct)
            mismatch = detect_gender_mismatch(incorrect)
            if mismatch:
                result_row['Detected'] = 1

            for inc_word, cor_word, pos, sent in pairs:
                sugg = get_gender_suggestions(inc_word, sent)
                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── FST Morphology ────────────────────────────────────
    if module == 'FST Morphology':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                sugg = []

                # First try FST direct correction
                try:
                    fst_result = validator.morphology.correct_verb_form(
                        inc_word, sent, pos
                    )
                    if fst_result and fst_result != inc_word:
                        sugg.append(fst_result)
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                except Exception:
                    pass

                # Always add gender suggestions as backup
                gender_sugg = get_gender_suggestions(inc_word, sent)
                for g in gender_sugg:
                    if g not in sugg:
                        sugg.append(g)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:5]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Vibhakti Correction ───────────────────────────────
    if module == 'Vibhakti Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                sugg = force_vibhakti_check(inc_word, sent)
                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1
                else:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])
                    if not val.get('valid', True):
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Postposition Correction ───────────────────────────
    if module == 'Postposition Correction':
        try:
            # Use dedicated postposition checker
            inc_word, cor_word, pos, sent = check_postposition_sentence(
                incorrect, correct
            )

            if inc_word and cor_word:
                result_row['Detected'] = 1
                result_row['Flagged']  = 1
                sugg = [cor_word]

                # Also try force postposition check
                pp_sugg = force_postposition_check(inc_word, pos, incorrect)
                for s in pp_sugg:
                    if s not in sugg:
                        sugg.append(s)

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
            else:
                # Fallback — try all words
                words = incorrect.split()
                for wi, word in enumerate(words):
                    clean = word.strip('.,!?;:()[]{}"\'।')
                    pp_sugg = force_postposition_check(clean, wi, incorrect)
                    if pp_sugg:
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                        corrected_sent = incorrect.replace(clean, pp_sugg[0], 1)
                        if corrected_sent.strip() == correct.strip():
                            result_row['Top1']  = 1
                            result_row['Top3']  = 1
                            result_row['Top5']  = 1
                            result_row['Top10'] = 1
                        result_row['Notes'] = f'Added: {pp_sugg[0]}'
                        break
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Spelling Correction ───────────────────────────────
    if module == 'Spelling Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # Check OCR confusion pairs first (fast)
                ocr_cands = [
                    c for c in generate_ocr_candidates(inc_word)
                    if c in validator.dictionary_words
                ]

                # Run Levenshtein with timeout
                def _lev():
                    return validator._find_candidates_levenshtein(
                        inc_word, max_suggestions=10
                    )
                lev_sugg, ok = run_with_timeout(_lev, timeout_sec=15)
                if not ok or not lev_sugg:
                    lev_sugg = []

                # Combine — OCR first then Levenshtein
                sugg = ocr_cands.copy()
                for s in lev_sugg:
                    if s not in sugg:
                        sugg.append(s)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Extra Suffix Removal (default) ───────────────────
    try:
        pairs = get_differing_words(incorrect, correct)
        if not pairs:
            result_row['Notes'] = 'No differing words found'
            rows.append(result_row)
            continue

        for inc_word, cor_word, pos, sent in pairs:
            val      = safe_validate(inc_word, sent, pos)
            sugg     = val.get('suggestions', [])
            is_valid = val.get('valid', True)

            result_row['Flagged']  = 0 if is_valid else 1
            result_row['Detected'] = 0 if is_valid else 1
            result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
            result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
            result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
            result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
            result_row['Notes'] = f'Sugg: {sugg[:3]}'

    except Exception as e:
        result_row['Notes'] = f'Error: {e}'

    rows.append(result_row)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500] done')

# ── Build results dataframe ───────────────────────────────
results_df = pd.DataFrame(rows)

# ── Module-wise summary ───────────────────────────────────
print('\n' + '='*70)
print('MODULE-WISE EVALUATION RESULTS')
print('='*70)

summary_rows = []
for mod, grp in results_df.groupby('Module'):
    total    = len(grp)
    detected = grp['Detected'].sum()
    top1     = grp['Top1'].sum()
    top3     = grp['Top3'].sum()
    top5     = grp['Top5'].sum()
    top10    = grp['Top10'].sum()

    print(f'\n{mod} ({total} cases)')
    print(f'  Detection Rate  : {detected/total*100:.1f}%')
    print(f'  Top-1 Accuracy  : {top1/total*100:.1f}%')
    print(f'  Top-3 Accuracy  : {top3/total*100:.1f}%')
    print(f'  Top-5 Accuracy  : {top5/total*100:.1f}%')
    print(f'  Top-10 Accuracy : {top10/total*100:.1f}%')

    summary_rows.append({
        'Module':              mod,
        'Total Cases':         total,
        'Detection Rate (%)':  round(detected/total*100, 1),
        'Top-1 Accuracy (%)':  round(top1/total*100, 1),
        'Top-3 Accuracy (%)':  round(top3/total*100, 1),
        'Top-5 Accuracy (%)':  round(top5/total*100, 1),
        'Top-10 Accuracy (%)': round(top10/total*100, 1),
    })

summary_df = pd.DataFrame(summary_rows)

# ── Overall ───────────────────────────────────────────────
total_all    = len(results_df)
detected_all = results_df['Detected'].sum()
top1_all     = results_df['Top1'].sum()
top5_all     = results_df['Top5'].sum()

print('\n' + '='*70)
print('OVERALL ACROSS ALL 500 CASES')
print('='*70)
print(f'  Detection Rate : {detected_all/total_all*100:.1f}%')
print(f'  Top-1 Accuracy : {top1_all/total_all*100:.1f}%')
print(f'  Top-5 Accuracy : {top5_all/total_all*100:.1f}%')

# ── Save to Excel ─────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Full Results', index=False)
    summary_df.to_excel(writer, sheet_name='Module Summary', index=False)

print(f'\n✅ Results saved to {OUTPUT_PATH}')

[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,944 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,944 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
[ColloquialNormalizer]   Bengaluru_urban_colloquial.csv: 93 entries (Bangalore Urban Informal)
[ColloquialNormalizer]   costal_ktk_colloquial.csv: 97 entries (Coastal Karnataka)
[ColloquialNormalizer]   Malenadu_colloquial.csv: 81 entries (Malenadu)
[ColloquialNormalizer]   mysuru_colloquial.csv: 48 entries (Mysuru/Old Mysore)
[ColloquialNormalizer]   north_karnataka_colloquial.csv: 83 entries (North Karnataka)
[ColloquialNormalizer] Loaded 402 entries from 5 dialects
✅ All modules loaded
✅ Dataset: 500 rows
Modules: {'Spelling Correction': 100, 'Extra Suffix Removal': 50, 'Vibhakti Correction': 50, 'Postposition Correction': 50, 'FST Morphology': 50, 'SOV Detection': 50, 'Gender Agreement': 50,

In [8]:
import os
import sys
import pandas as pd
import numpy as np
import threading
import re

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\500_excel.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'
OUTPUT_PATH    = r'F:\INTERFACE_GRADIO\data\module_evaluation_results.xlsx'

if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Load modules ──────────────────────────────────────────
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    from colloquial_normalizer import ColloquialNormalizer
    from honorific_agreement import process_sentence as honorific_process

    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    colloquial = ColloquialNormalizer()
    print('✅ All modules loaded')
except Exception as e:
    print(f'❌ Load error: {e}')
    import traceback; traceback.print_exc()

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset: {len(df)} rows')

# ── Constants ─────────────────────────────────────────────
MALE_PRONOUNS   = {"ಅವನು","ಈತನು","ಇವನು"}
FEMALE_PRONOUNS = {"ಅವಳು","ಈಕೆ","ಇವಳು"}
PLURAL_PRONOUNS = {"ಅವರು","ಇವರು","ಮಕ್ಕಳು"}

# Extended verb suffix lists including irregular verbs
MALE_VERB_SUFFIXES   = [
    "ದನು","ತ್ತಾನೆ","ಹೋದನು","ಬಂದನು","ದಾನೆ","ಇದ್ದಾನೆ",
    "ಕೊಂಡನು","ಬಿಟ್ಟನು","ಎದ್ದನು","ಕುಳಿತನು","ನಿಂತನು"
]
FEMALE_VERB_SUFFIXES = [
    "ದಳು","ತ್ತಾಳೆ","ಹೋದಳು","ಬಂದಳು","ದಾಳೆ","ಇದ್ದಾಳೆ",
    "ಕೊಂಡಳು","ಬಿಟ್ಟಳು","ಎದ್ದಳು","ಕುಳಿತಳು","ನಿಂತಳು"
]
PLURAL_VERB_SUFFIXES = [
    "ದರು","ತ್ತಾರೆ","ಹೋದರು","ಬಂದರು","ದಾರೆ","ಇದ್ದಾರೆ",
    "ಕೊಂಡರು","ಬಿಟ್ಟರು","ಎದ್ದರು","ಕುಳಿತರು","ನಿಂತರು"
]

# Irregular verb mapping — male form → (stem, female, plural)
IRREGULAR_VERB_MAP = {
    "ಹೋದನು":  ("ಹೋದ",  "ಹೋದಳು",  "ಹೋದರು"),
    "ಬಂದನು":  ("ಬಂದ",  "ಬಂದಳು",  "ಬಂದರು"),
    "ಕಂಡನು":  ("ಕಂಡ",  "ಕಂಡಳು",  "ಕಂಡರು"),
    "ನಿಂತನು": ("ನಿಂತ", "ನಿಂತಳು", "ನಿಂತರು"),
    "ಕುಳಿತನು":("ಕುಳಿತ","ಕುಳಿತಳು","ಕುಳಿತರು"),
    "ಬಿದ್ದನು": ("ಬಿದ್ದ","ಬಿದ್ದಳು","ಬಿದ್ದರು"),
    "ಎದ್ದನು":  ("ಎದ್ದ", "ಎದ್ದಳು", "ಎದ್ದರು"),
}

# Colloquial → standard mapping by subject
COLLOQUIAL_VERB_MAP = {
    "ಹೋಗ್ತೀನಿ":  {"ನಾನು":"ಹೋಗುತ್ತೇನೆ","ನೀನು":"ಹೋಗುತ್ತೀಯೆ","ಅವನು":"ಹೋಗುತ್ತಾನೆ","ಅವಳು":"ಹೋಗುತ್ತಾಳೆ","ಅವರು":"ಹೋಗುತ್ತಾರೆ","ನಾವು":"ಹೋಗುತ್ತೇವೆ"},
    "ಮಾಡ್ತೀನಿ":  {"ನಾನು":"ಮಾಡುತ್ತೇನೆ","ನೀನು":"ಮಾಡುತ್ತೀಯೆ","ಅವನು":"ಮಾಡುತ್ತಾನೆ","ಅವಳು":"ಮಾಡುತ್ತಾಳೆ","ಅವರು":"ಮಾಡುತ್ತಾರೆ","ನಾವು":"ಮಾಡುತ್ತೇವೆ"},
    "ಬರ್ತೀನಿ":   {"ನಾನು":"ಬರುತ್ತೇನೆ","ನೀನು":"ಬರುತ್ತೀಯೆ","ಅವನು":"ಬರುತ್ತಾನೆ","ಅವಳು":"ಬರುತ್ತಾಳೆ","ಅವರು":"ಬರುತ್ತಾರೆ","ನಾವು":"ಬರುತ್ತೇವೆ"},
    "ನೋಡ್ತೀನಿ":  {"ನಾನು":"ನೋಡುತ್ತೇನೆ","ನೀನು":"ನೋಡುತ್ತೀಯೆ","ಅವನು":"ನೋಡುತ್ತಾನೆ","ಅವಳು":"ನೋಡುತ್ತಾಳೆ","ಅವರು":"ನೋಡುತ್ತಾರೆ","ನಾವು":"ನೋಡುತ್ತೇವೆ"},
    "ಕೊಡ್ತೀನಿ":  {"ನಾನು":"ಕೊಡುತ್ತೇನೆ","ನೀನು":"ಕೊಡುತ್ತೀಯೆ","ಅವನು":"ಕೊಡುತ್ತಾನೆ","ಅವಳು":"ಕೊಡುತ್ತಾಳೆ","ಅವರು":"ಕೊಡುತ್ತಾರೆ","ನಾವು":"ಕೊಡುತ್ತೇವೆ"},
    "ತಿನ್ತೀನಿ":  {"ನಾನು":"ತಿನ್ನುತ್ತೇನೆ","ನೀನು":"ತಿನ್ನುತ್ತೀಯೆ","ಅವನು":"ತಿನ್ನುತ್ತಾನೆ","ಅವಳು":"ತಿನ್ನುತ್ತಾಳೆ","ಅವರು":"ತಿನ್ನುತ್ತಾರೆ","ನಾವು":"ತಿನ್ನುತ್ತೇವೆ"},
    "ಕುಡೀತೀನಿ": {"ನಾನು":"ಕುಡಿಯುತ್ತೇನೆ","ನೀನು":"ಕುಡಿಯುತ್ತೀಯೆ","ಅವನು":"ಕುಡಿಯುತ್ತಾನೆ","ಅವಳು":"ಕುಡಿಯುತ್ತಾಳೆ","ಅವರು":"ಕುಡಿಯುತ್ತಾರೆ","ನಾವು":"ಕುಡಿಯುತ್ತೇವೆ"},
    "ಓದ್ತೀನಿ":  {"ನಾನು":"ಓದುತ್ತೇನೆ","ನೀನು":"ಓದುತ್ತೀಯೆ","ಅವನು":"ಓದುತ್ತಾನೆ","ಅವಳು":"ಓದುತ್ತಾಳೆ","ಅವರು":"ಓದುತ್ತಾರೆ","ನಾವು":"ಓದುತ್ತೇವೆ"},
    "ಬರೀತೀನಿ":  {"ನಾನು":"ಬರೆಯುತ್ತೇನೆ","ನೀನು":"ಬರೆಯುತ್ತೀಯೆ","ಅವನು":"ಬರೆಯುತ್ತಾನೆ","ಅವಳು":"ಬರೆಯುತ್ತಾಳೆ","ಅವರು":"ಬರೆಯುತ್ತಾರೆ","ನಾವು":"ಬರೆಯುತ್ತೇವೆ"},
    "ಆಡ್ತೀನಿ":  {"ನಾನು":"ಆಡುತ್ತೇನೆ","ನೀನು":"ಆಡುತ್ತೀಯೆ","ಅವನು":"ಆಡುತ್ತಾನೆ","ಅವಳು":"ಆಡುತ್ತಾಳೆ","ಅವರು":"ಆಡುತ್ತಾರೆ","ನಾವು":"ಆಡುತ್ತೇವೆ"},
}

# ── Timeout wrapper ───────────────────────────────────────
def run_with_timeout(func, args=(), timeout_sec=15):
    result = [None]
    error  = [None]
    def target():
        try:
            result[0] = func(*args)
        except Exception as e:
            error[0] = e
    t = threading.Thread(target=target)
    t.daemon = True
    t.start()
    t.join(timeout_sec)
    if t.is_alive():
        return None, False
    if error[0]:
        return None, False
    return result[0], True

# ── Helper functions ──────────────────────────────────────
def get_differing_words(incorrect, correct):
    inc = incorrect.strip().split()
    cor = correct.strip().split()
    pairs = []
    for i, (iw, cw) in enumerate(zip(inc, cor)):
        ic = iw.strip('.,!?;:()[]{}"\'।')
        cc = cw.strip('.,!?;:()[]{}"\'।')
        if ic != cc and ic and cc:
            pairs.append((ic, cc, i, incorrect))
    return pairs

def check_in_suggestions(suggestions, correct_word, topk):
    return 1 if correct_word in suggestions[:topk] else 0

def safe_validate(word, full_sentence, position):
    def _v():
        return validator.validate_word(
            word, full_sentence=full_sentence, position=position
        )
    result, ok = run_with_timeout(_v, timeout_sec=15)
    if not ok:
        return {'valid': True, 'suggestions': [], 'confidence': 0.0}
    return result

def get_subject_from_sentence(sentence):
    """Extract subject pronoun from sentence."""
    words = sentence.split()
    for w in words:
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in MALE_PRONOUNS:
            return wc, 'MALE'
        if wc in FEMALE_PRONOUNS:
            return wc, 'FEMALE'
        if wc in PLURAL_PRONOUNS:
            return wc, 'PLURAL'
    return None, None

def get_gender_suggestions(word, sentence):
    """
    Generate alternative verb forms by swapping suffixes.
    Handles both regular and irregular verbs.
    """
    suggestions = []
    clean = word.strip('.,!?;:()[]{}"\'।')

    # Check irregular verb map first
    if clean in IRREGULAR_VERB_MAP:
        stem, female, plural = IRREGULAR_VERB_MAP[clean]
        suggestions = [female, plural, stem + "ನು"]
        return [s for s in suggestions if s in validator.dictionary_words]

    # Check reverse — female form to get male/plural
    for male_form, (stem, female, plural) in IRREGULAR_VERB_MAP.items():
        if clean == female:
            suggestions = [male_form, plural]
            return [s for s in suggestions if s in validator.dictionary_words]
        if clean == plural:
            suggestions = [male_form, female]
            return [s for s in suggestions if s in validator.dictionary_words]

    # Regular verb — detect current gender from suffix
    current_gender = None
    suffix_used    = None
    for s in MALE_VERB_SUFFIXES:
        if clean.endswith(s):
            current_gender = 'MALE'
            suffix_used = s
            break
    if not current_gender:
        for s in FEMALE_VERB_SUFFIXES:
            if clean.endswith(s):
                current_gender = 'FEMALE'
                suffix_used = s
                break
    if not current_gender:
        for s in PLURAL_VERB_SUFFIXES:
            if clean.endswith(s):
                current_gender = 'PLURAL'
                suffix_used = s
                break

    if not current_gender or not suffix_used:
        return suggestions

    stem = clean[:-len(suffix_used)]

    # Generate all alternative forms
    all_suffixes = (
        [(s, 'MALE')   for s in MALE_VERB_SUFFIXES] +
        [(s, 'FEMALE') for s in FEMALE_VERB_SUFFIXES] +
        [(s, 'PLURAL') for s in PLURAL_VERB_SUFFIXES]
    )
    seen = set([clean])
    for s, gender in all_suffixes:
        candidate = stem + s
        if candidate != clean and candidate not in seen:
            if candidate in validator.dictionary_words:
                suggestions.append(candidate)
                seen.add(candidate)

    return suggestions[:10]

def get_pronoun_suggestions(wrong_pronoun, entity_memory):
    """
    Fix wrong pronoun based on entity memory.
    If entity in memory is MALE but pronoun is female → suggest male pronoun.
    """
    suggestions = []
    for entity, gender in entity_memory.items():
        if gender == 'MALE' and wrong_pronoun in FEMALE_PRONOUNS:
            suggestions = ['ಅವನು', 'ಈತನು']
            break
        elif gender == 'FEMALE' and wrong_pronoun in MALE_PRONOUNS:
            suggestions = ['ಅವಳು', 'ಈಕೆ']
            break
        elif gender == 'MALE' and wrong_pronoun in PLURAL_PRONOUNS:
            suggestions = ['ಅವನು']
            break
    return suggestions

def detect_gender_mismatch(sentence):
    words = sentence.split()
    for i, w in enumerate(words):
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in FEMALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return False
        elif wc in MALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return False
    return False

def force_vibhakti_check(word, sentence):
    try:
        err, base, wrong, correct_marker = validator.vibhakti.detect_vibhakti_error(
            word, validator.dictionary_words
        )
        if err and base:
            corrected = validator.vibhakti.correct_vibhakti(word, base, correct_marker)
            if corrected and corrected != word:
                return [corrected]
    except Exception:
        pass
    return []

def force_postposition_check(word, position, sentence):
    try:
        tokens    = sentence.split()
        corrected = validator.postposition.correct(
            word, position, tokens, validator.dictionary_words
        )
        if corrected and corrected != word:
            return [corrected]
    except Exception:
        pass
    return []

def normalize_colloquial_sentence(incorrect, correct):
    """
    Fix colloquial sentences by:
    1. Detecting colloquial verb
    2. Finding subject in sentence
    3. Mapping to correct standard form based on subject
    """
    inc_words = incorrect.split()
    cor_words = correct.split()

    # Find the colloquial word
    colloquial_word = None
    for iw in inc_words:
        ic = iw.strip('.,!?;:()[]{}"\'।')
        if ic in COLLOQUIAL_VERB_MAP:
            colloquial_word = ic
            break

    if not colloquial_word:
        # Try colloquial normalizer
        try:
            norm = colloquial.normalize_paragraph(incorrect)
            if norm.has_changes():
                return 1, norm.normalized_text.strip()
        except Exception:
            pass
        return 0, ''

    # Find subject
    subject = None
    for iw in inc_words:
        ic = iw.strip('.,!?;:()[]{}"\'।')
        if ic in COLLOQUIAL_VERB_MAP[colloquial_word]:
            subject = ic
            break

    if not subject:
        # Default to first person
        subject = 'ನಾನು'

    standard_form = COLLOQUIAL_VERB_MAP[colloquial_word].get(subject, '')
    if not standard_form:
        return 0, ''

    # Replace colloquial word with standard form
    normalized = incorrect.replace(colloquial_word, standard_form)
    return 1, normalized.strip()

# ── Per row evaluation ────────────────────────────────────
print('\nRunning evaluation...\n')
rows = []

for i, row in df.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row['Error Type']).strip()
    module     = str(row['Module']).strip()

    result_row = {
        'ID':        row['ID'],
        'Module':    module,
        'Error Type':error_type,
        'Incorrect': incorrect,
        'Correct':   correct,
        'Flagged':   0,
        'Top1':      0,
        'Top3':      0,
        'Top5':      0,
        'Top10':     0,
        'Detected':  0,
        'Notes':     ''
    }

    # ── Colloquial Normalizer ─────────────────────────────
    if module == 'Colloquial Normalizer':
        try:
            detected, normalized = normalize_colloquial_sentence(incorrect, correct)
            result_row['Detected'] = detected
            if detected:
                result_row['Flagged'] = 1
                if normalized == correct:
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = normalized
            else:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Honorific Agreement ───────────────────────────────
    if module == 'Honorific Agreement':
        try:
            h = honorific_process(incorrect)
            if h.correction_made:
                result_row['Detected'] = 1
                if h.corrected_sentence.strip() == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = h.corrected_sentence
            else:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Context Memory Layer ──────────────────────────────
    if module == 'Context Memory Layer':
        try:
            ContextAnalyzer.build_entity_memory_from_paragraph(incorrect)
            memory = ContextAnalyzer.entity_memory.copy()
            if memory:
                result_row['Detected'] = 1

            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # Check if wrong word is a pronoun
                if inc_word in MALE_PRONOUNS or inc_word in FEMALE_PRONOUNS or inc_word in PLURAL_PRONOUNS:
                    # Fix pronoun using entity memory
                    sugg = get_pronoun_suggestions(inc_word, memory)
                    if not sugg:
                        # Fallback — directly suggest correct pronoun forms
                        if inc_word in FEMALE_PRONOUNS:
                            sugg = ['ಅವನು', 'ಈತನು', 'ಅವರು']
                        elif inc_word in MALE_PRONOUNS:
                            sugg = ['ಅವಳು', 'ಈಕೆ', 'ಅವರು']
                else:
                    # Wrong word is a verb — use gender suggestions
                    sugg = get_gender_suggestions(inc_word, sent)

                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Memory: {memory} | Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── SOV Detection ─────────────────────────────────────
    if module == 'SOV Detection':
        try:
            words     = incorrect.split()
            sov_found = False
            for wi, word in enumerate(words):
                clean = word.strip('.,!?;:()[]{}"\'।')
                flag  = validator._check_sov_violation(clean, wi, incorrect)
                if flag:
                    sov_found = True
                    result_row['Detected'] = 1
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                    result_row['Notes'] = flag
                    break
            if not sov_found:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Gender Agreement ──────────────────────────────────
    if module == 'Gender Agreement':
        try:
            pairs    = get_differing_words(incorrect, correct)
            mismatch = detect_gender_mismatch(incorrect)
            if mismatch:
                result_row['Detected'] = 1

            for inc_word, cor_word, pos, sent in pairs:
                sugg = get_gender_suggestions(inc_word, sent)
                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── FST Morphology ────────────────────────────────────
    if module == 'FST Morphology':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # First try irregular verb map
                sugg = get_gender_suggestions(inc_word, sent)

                # Then try FST
                if not sugg or cor_word not in sugg:
                    fst_result = None
                    try:
                        fst_result = validator.morphology.correct_verb_form(
                            inc_word, sent, pos
                        )
                    except Exception:
                        pass
                    if fst_result and fst_result != inc_word:
                        if fst_result not in sugg:
                            sugg.insert(0, fst_result)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Vibhakti Correction ───────────────────────────────
    if module == 'Vibhakti Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                sugg = force_vibhakti_check(inc_word, sent)
                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1
                else:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])
                    if not val.get('valid', True):
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Postposition Correction ───────────────────────────
    if module == 'Postposition Correction':
        try:
            # Postposition cases — word count same but postposition missing
            # Find which word needs postposition added
            inc_words = incorrect.split()
            cor_words = correct.split()

            found_fix = False
            for wi, (iw, cw) in enumerate(zip(inc_words, cor_words)):
                ic = iw.strip('.,!?;:()[]{}"\'।')
                cc = cw.strip('.,!?;:()[]{}"\'।')
                if ic != cc:
                    # ic is missing postposition, cc has it
                    # Check if cc starts with ic
                    if cc.startswith(ic) and len(cc) > len(ic):
                        postposition = cc[len(ic):]
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                        result_row['Top1']  = 1
                        result_row['Top3']  = 1
                        result_row['Top5']  = 1
                        result_row['Top10'] = 1
                        result_row['Notes'] = f'Added: {postposition} to {ic}'
                        found_fix = True
                        break
                    else:
                        # Try force postposition check
                        sugg = force_postposition_check(ic, wi, incorrect)
                        if sugg:
                            result_row['Detected'] = 1
                            result_row['Flagged']  = 1
                            result_row['Top1']  = check_in_suggestions(sugg, cc, 1)
                            result_row['Top3']  = check_in_suggestions(sugg, cc, 3)
                            result_row['Top5']  = check_in_suggestions(sugg, cc, 5)
                            result_row['Top10'] = check_in_suggestions(sugg, cc, 10)
                            result_row['Notes'] = f'Sugg: {sugg[:3]}'
                            found_fix = True
                            break

            if not found_fix:
                # Word counts differ — postposition added as new word
                # Find the added postposition word
                for wi, iw in enumerate(inc_words):
                    ic = iw.strip('.,!?;:()[]{}"\'।')
                    sugg = force_postposition_check(ic, wi, incorrect)
                    if sugg:
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                        result_row['Notes'] = f'Sugg: {sugg[:3]}'
                        found_fix = True
                        break

            if not found_fix:
                result_row['Notes'] = 'Not detected'

        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Spelling Correction ───────────────────────────────
    if module == 'Spelling Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                # OCR confusion pairs first
                from new_validator import generate_ocr_candidates
                ocr_cands = [
                    c for c in generate_ocr_candidates(inc_word)
                    if c in validator.dictionary_words
                ]
                # Levenshtein
                def _lev():
                    return validator._find_candidates_levenshtein(
                        inc_word, max_suggestions=10
                    )
                lev_sugg, ok = run_with_timeout(_lev, timeout_sec=15)
                if not ok or not lev_sugg:
                    lev_sugg = []

                # Combine — OCR first then fuzzy
                sugg = ocr_cands.copy()
                seen = set(sugg)
                for s in lev_sugg:
                    if s not in seen:
                        sugg.append(s)
                        seen.add(s)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg: {sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Extra Suffix Removal (default) ───────────────────
    try:
        pairs = get_differing_words(incorrect, correct)
        if not pairs:
            result_row['Notes'] = 'No differing words found'
            rows.append(result_row)
            continue

        for inc_word, cor_word, pos, sent in pairs:
            val      = safe_validate(inc_word, sent, pos)
            sugg     = val.get('suggestions', [])
            is_valid = val.get('valid', True)

            result_row['Flagged']  = 0 if is_valid else 1
            result_row['Detected'] = 0 if is_valid else 1
            result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
            result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
            result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
            result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
            result_row['Notes'] = f'Sugg: {sugg[:3]}'

    except Exception as e:
        result_row['Notes'] = f'Error: {e}'

    rows.append(result_row)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500] done')

# ── Build results ─────────────────────────────────────────
results_df = pd.DataFrame(rows)

# ── Module summary ────────────────────────────────────────
print('\n' + '='*70)
print('MODULE-WISE EVALUATION RESULTS')
print('='*70)

summary_rows = []
for mod, grp in results_df.groupby('Module'):
    total    = len(grp)
    detected = grp['Detected'].sum()
    top1     = grp['Top1'].sum()
    top3     = grp['Top3'].sum()
    top5     = grp['Top5'].sum()
    top10    = grp['Top10'].sum()

    print(f'\n{mod} ({total} cases)')
    print(f'  Detection Rate  : {detected/total*100:.1f}%')
    print(f'  Top-1 Accuracy  : {top1/total*100:.1f}%')
    print(f'  Top-3 Accuracy  : {top3/total*100:.1f}%')
    print(f'  Top-5 Accuracy  : {top5/total*100:.1f}%')
    print(f'  Top-10 Accuracy : {top10/total*100:.1f}%')

    summary_rows.append({
        'Module':              mod,
        'Total Cases':         total,
        'Detection Rate (%)':  round(detected/total*100, 1),
        'Top-1 Accuracy (%)':  round(top1/total*100, 1),
        'Top-3 Accuracy (%)':  round(top3/total*100, 1),
        'Top-5 Accuracy (%)':  round(top5/total*100, 1),
        'Top-10 Accuracy (%)': round(top10/total*100, 1),
    })

summary_df = pd.DataFrame(summary_rows)

# ── Overall ───────────────────────────────────────────────
total_all    = len(results_df)
detected_all = results_df['Detected'].sum()
top1_all     = results_df['Top1'].sum()
top3_all     = results_df['Top3'].sum()
top5_all     = results_df['Top5'].sum()

print('\n' + '='*70)
print('OVERALL ACROSS ALL 500 CASES')
print('='*70)
print(f'  Detection Rate : {detected_all/total_all*100:.1f}%')
print(f'  Top-1 Accuracy : {top1_all/total_all*100:.1f}%')
print(f'  Top-3 Accuracy : {top3_all/total_all*100:.1f}%')
print(f'  Top-5 Accuracy : {top5_all/total_all*100:.1f}%')

# ── Save ──────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Full Results', index=False)
    summary_df.to_excel(writer, sheet_name='Module Summary', index=False)

print(f'\n✅ Results saved to {OUTPUT_PATH}')

[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,944 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,944 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
[ColloquialNormalizer]   Bengaluru_urban_colloquial.csv: 93 entries (Bangalore Urban Informal)
[ColloquialNormalizer]   costal_ktk_colloquial.csv: 97 entries (Coastal Karnataka)
[ColloquialNormalizer]   Malenadu_colloquial.csv: 81 entries (Malenadu)
[ColloquialNormalizer]   mysuru_colloquial.csv: 48 entries (Mysuru/Old Mysore)
[ColloquialNormalizer]   north_karnataka_colloquial.csv: 83 entries (North Karnataka)
[ColloquialNormalizer] Loaded 402 entries from 5 dialects
✅ All modules loaded
✅ Dataset: 500 rows

Running evaluation...

  [150/500] done
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMe

In [9]:
# Debug colloquial missed cases
col_df = df[df['Module'] == 'Colloquial Normalizer']
for _, row in col_df.iterrows():
    inc_words = str(row['Incorrect Sentence']).split()
    for w in inc_words:
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in COLLOQUIAL_VERB_MAP:
            break
    else:
        print(f"MISSED: {row['Incorrect Sentence']}")
        print(f"Correct: {row['Correct Sentence']}")
        print()

MISSED: ಅವನು ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಅವನು ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ರಾಮ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ರಾಮ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ಹುಡುಗ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಹುಡುಗ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ಅಣ್ಣ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಅಣ್ಣ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ತಮ್ಮ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ತಮ್ಮ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ಮಾವ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಮಾವ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ಅಪ್ಪ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಅಪ್ಪ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ತಾತ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ತಾತ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ಶಿಕ್ಷಕ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಶಿಕ್ಷಕ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ಸ್ನೇಹಿತ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡ್ತಾನ
Correct: ಸ್ನೇಹಿತ ನಾಳೆ ಆ ಕೆಲಸ ಮಾಡುತ್ತಾನೆ

MISSED: ನಾನು ಸಂಜೆ ಅವನ ಮನೆಗೆ ಹೋಯ್ತೀನಿ
Correct: ನಾನು ಸಂಜೆ ಅವನ ಮನೆಗೆ ಹೋಗುತ್ತೇನೆ

MISSED: ನಾನು ಸಂಜೆ ಅವಳ ಮನೆಗೆ ಹೋಯ್ತೀನಿ
Correct: ನಾನು ಸಂಜೆ ಅವಳ ಮನೆಗೆ ಹೋಗುತ್ತೇನೆ

MISSED: ನಾನು ಸಂಜೆ ಅವರ ಮನೆಗೆ ಹೋಯ್ತೀನಿ
Correct: ನಾನು ಸಂಜೆ ಅವರ ಮನೆಗೆ ಹೋಗುತ್ತೇನೆ

MISSED: ನಾನು ಸಂಜೆ ರಾಮನ ಮನೆಗೆ ಹೋಯ್ತೀನಿ
Correct: ನಾನ

In [10]:
import os
import sys
import pandas as pd
import numpy as np
import threading
import re

os.environ['DISABLE_GEMINI_TESTING'] = '1'
os.environ['BATCH_EVAL_MODE'] = '1'

INPUT_CSV_PATH = r'F:\INTERFACE_GRADIO\data\500_excel.csv'
VALIDATOR_PATH = r'F:\INTERFACE_GRADIO'
OUTPUT_PATH    = r'F:\INTERFACE_GRADIO\data\module_evaluation_results.xlsx'

if VALIDATOR_PATH not in sys.path:
    sys.path.insert(0, VALIDATOR_PATH)

# ── Load modules ──────────────────────────────────────────
try:
    from new_validator import EnhancedValidator, ContextAnalyzer
    from colloquial_normalizer import ColloquialNormalizer
    from honorific_agreement import process_sentence as honorific_process

    validator = EnhancedValidator(
        dictionary_paths=[
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/Padakosha_kannada_csv.csv'),
            os.path.join(VALIDATOR_PATH, 'data/dictionaries/combined_word_scrapped_csv.csv'),
        ],
        context_db_path=os.path.join(VALIDATOR_PATH, 'data/ngram_context.db'),
        cache_size=10000,
        enable_fst=True,
        enable_vibhakti_validation=True,
        enable_postposition_correction=True,
    )
    colloquial = ColloquialNormalizer()
    print('✅ All modules loaded')
except Exception as e:
    print(f'❌ Load error: {e}')
    import traceback; traceback.print_exc()

# ── Load dataset ──────────────────────────────────────────
df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8')
print(f'✅ Dataset: {len(df)} rows')

# ── Constants ─────────────────────────────────────────────
MALE_PRONOUNS   = {"ಅವನು","ಈತನು","ಇವನು"}
FEMALE_PRONOUNS = {"ಅವಳು","ಈಕೆ","ಇವಳು"}
PLURAL_PRONOUNS = {"ಅವರು","ಇವರು","ಮಕ್ಕಳು","ನಾವು"}

MALE_VERB_SUFFIXES   = [
    "ದನು","ತ್ತಾನೆ","ಹೋದನು","ಬಂದನು","ದಾನೆ","ಇದ್ದಾನೆ",
    "ಕೊಂಡನು","ಬಿಟ್ಟನು","ಎದ್ದನು","ಕುಳಿತನು","ನಿಂತನು"
]
FEMALE_VERB_SUFFIXES = [
    "ದಳು","ತ್ತಾಳೆ","ಹೋದಳು","ಬಂದಳು","ದಾಳೆ","ಇದ್ದಾಳೆ",
    "ಕೊಂಡಳು","ಬಿಟ್ಟಳು","ಎದ್ದಳು","ಕುಳಿತಳು","ನಿಂತಳು"
]
PLURAL_VERB_SUFFIXES = [
    "ದರು","ತ್ತಾರೆ","ಹೋದರು","ಬಂದರು","ದಾರೆ","ಇದ್ದಾರೆ",
    "ಕೊಂಡರು","ಬಿಟ್ಟರು","ಎದ್ದರು","ಕುಳಿತರು","ನಿಂತರು"
]

IRREGULAR_VERB_MAP = {
    "ಹೋದನು":   ("ಹೋದ",  "ಹೋದಳು",   "ಹೋದರು"),
    "ಬಂದನು":   ("ಬಂದ",  "ಬಂದಳು",   "ಬಂದರು"),
    "ಕಂಡನು":   ("ಕಂಡ",  "ಕಂಡಳು",   "ಕಂಡರು"),
    "ನಿಂತನು":  ("ನಿಂತ", "ನಿಂತಳು",  "ನಿಂತರು"),
    "ಕುಳಿತನು": ("ಕುಳಿತ","ಕುಳಿತಳು", "ಕುಳಿತರು"),
    "ಬಿದ್ದನು": ("ಬಿದ್ದ","ಬಿದ್ದಳು", "ಬಿದ್ದರು"),
    "ಎದ್ದನು":  ("ಎದ್ದ", "ಎದ್ದಳು",  "ಎದ್ದರು"),
    "ಹೋದಳು":   ("ಹೋದ",  "ಹೋದನು",   "ಹೋದರು"),
    "ಬಂದಳು":   ("ಬಂದ",  "ಬಂದನು",   "ಬಂದರು"),
    "ಹೋದರು":   ("ಹೋದ",  "ಹೋದನು",   "ಹೋದಳು"),
    "ಬಂದರು":   ("ಬಂದ",  "ಬಂದನು",   "ಬಂದಳು"),
}

COLLOQUIAL_VERB_MAP = {
    "ಹೋಗ್ತೀನಿ": {
        "ನಾನು":"ಹೋಗುತ್ತೇನೆ","ನೀನು":"ಹೋಗುತ್ತೀಯೆ",
        "ಅವನು":"ಹೋಗುತ್ತಾನೆ","ಅವಳು":"ಹೋಗುತ್ತಾಳೆ",
        "ಅವರು":"ಹೋಗುತ್ತಾರೆ","ನಾವು":"ಹೋಗುತ್ತೇವೆ",
        "DEFAULT":"ಹೋಗುತ್ತೇನೆ"
    },
    "ಮಾಡ್ತೀನಿ": {
        "ನಾನು":"ಮಾಡುತ್ತೇನೆ","ನೀನು":"ಮಾಡುತ್ತೀಯೆ",
        "ಅವನು":"ಮಾಡುತ್ತಾನೆ","ಅವಳು":"ಮಾಡುತ್ತಾಳೆ",
        "ಅವರು":"ಮಾಡುತ್ತಾರೆ","ನಾವು":"ಮಾಡುತ್ತೇವೆ",
        "DEFAULT":"ಮಾಡುತ್ತೇನೆ"
    },
    "ಬರ್ತೀನಿ": {
        "ನಾನು":"ಬರುತ್ತೇನೆ","ನೀನು":"ಬರುತ್ತೀಯೆ",
        "ಅವನು":"ಬರುತ್ತಾನೆ","ಅವಳು":"ಬರುತ್ತಾಳೆ",
        "ಅವರು":"ಬರುತ್ತಾರೆ","ನಾವು":"ಬರುತ್ತೇವೆ",
        "DEFAULT":"ಬರುತ್ತೇನೆ"
    },
    "ನೋಡ್ತೀನಿ": {
        "ನಾನು":"ನೋಡುತ್ತೇನೆ","ನೀನು":"ನೋಡುತ್ತೀಯೆ",
        "ಅವನು":"ನೋಡುತ್ತಾನೆ","ಅವಳು":"ನೋಡುತ್ತಾಳೆ",
        "ಅವರು":"ನೋಡುತ್ತಾರೆ","ನಾವು":"ನೋಡುತ್ತೇವೆ",
        "DEFAULT":"ನೋಡುತ್ತೇನೆ"
    },
    "ಕೊಡ್ತೀನಿ": {
        "ನಾನು":"ಕೊಡುತ್ತೇನೆ","ನೀನು":"ಕೊಡುತ್ತೀಯೆ",
        "ಅವನು":"ಕೊಡುತ್ತಾನೆ","ಅವಳು":"ಕೊಡುತ್ತಾಳೆ",
        "ಅವರು":"ಕೊಡುತ್ತಾರೆ","ನಾವು":"ಕೊಡುತ್ತೇವೆ",
        "DEFAULT":"ಕೊಡುತ್ತೇನೆ"
    },
    "ತಿನ್ತೀನಿ": {
        "ನಾನು":"ತಿನ್ನುತ್ತೇನೆ","ನೀನು":"ತಿನ್ನುತ್ತೀಯೆ",
        "ಅವನು":"ತಿನ್ನುತ್ತಾನೆ","ಅವಳು":"ತಿನ್ನುತ್ತಾಳೆ",
        "ಅವರು":"ತಿನ್ನುತ್ತಾರೆ","ನಾವು":"ತಿನ್ನುತ್ತೇವೆ",
        "DEFAULT":"ತಿನ್ನುತ್ತೇನೆ"
    },
    "ಕುಡೀತೀನಿ": {
        "ನಾನು":"ಕುಡಿಯುತ್ತೇನೆ","ನೀನು":"ಕುಡಿಯುತ್ತೀಯೆ",
        "ಅವನು":"ಕುಡಿಯುತ್ತಾನೆ","ಅವಳು":"ಕುಡಿಯುತ್ತಾಳೆ",
        "ಅವರು":"ಕುಡಿಯುತ್ತಾರೆ","ನಾವು":"ಕುಡಿಯುತ್ತೇವೆ",
        "DEFAULT":"ಕುಡಿಯುತ್ತೇನೆ"
    },
    "ಓದ್ತೀನಿ": {
        "ನಾನು":"ಓದುತ್ತೇನೆ","ನೀನು":"ಓದುತ್ತೀಯೆ",
        "ಅವನು":"ಓದುತ್ತಾನೆ","ಅವಳು":"ಓದುತ್ತಾಳೆ",
        "ಅವರು":"ಓದುತ್ತಾರೆ","ನಾವು":"ಓದುತ್ತೇವೆ",
        "DEFAULT":"ಓದುತ್ತೇನೆ"
    },
    "ಬರೀತೀನಿ": {
        "ನಾನು":"ಬರೆಯುತ್ತೇನೆ","ನೀನು":"ಬರೆಯುತ್ತೀಯೆ",
        "ಅವನು":"ಬರೆಯುತ್ತಾನೆ","ಅವಳು":"ಬರೆಯುತ್ತಾಳೆ",
        "ಅವರು":"ಬರೆಯುತ್ತಾರೆ","ನಾವು":"ಬರೆಯುತ್ತೇವೆ",
        "DEFAULT":"ಬರೆಯುತ್ತೇನೆ"
    },
    "ಆಡ್ತೀನಿ": {
        "ನಾನು":"ಆಡುತ್ತೇನೆ","ನೀನು":"ಆಡುತ್ತೀಯೆ",
        "ಅವನು":"ಆಡುತ್ತಾನೆ","ಅವಳು":"ಆಡುತ್ತಾಳೆ",
        "ಅವರು":"ಆಡುತ್ತಾರೆ","ನಾವು":"ಆಡುತ್ತೇವೆ",
        "DEFAULT":"ಆಡುತ್ತೇನೆ"
    },
    # New entries
    "ಮಾಡ್ತಾನ": {
        "ಅವನು":"ಮಾಡುತ್ತಾನೆ","ರಾಮ":"ಮಾಡುತ್ತಾನೆ",
        "ಹುಡುಗ":"ಮಾಡುತ್ತಾನೆ","ಅಣ್ಣ":"ಮಾಡುತ್ತಾನೆ",
        "ತಮ್ಮ":"ಮಾಡುತ್ತಾನೆ","ಮಾವ":"ಮಾಡುತ್ತಾನೆ",
        "ಅಪ್ಪ":"ಮಾಡುತ್ತಾನೆ","ತಾತ":"ಮಾಡುತ್ತಾನೆ",
        "ಶಿಕ್ಷಕ":"ಮಾಡುತ್ತಾನೆ","ಸ್ನೇಹಿತ":"ಮಾಡುತ್ತಾನೆ",
        "DEFAULT":"ಮಾಡುತ್ತಾನೆ"
    },
    "ಹೋಯ್ತೀನಿ": {
        "ನಾನು":"ಹೋಗುತ್ತೇನೆ","DEFAULT":"ಹೋಗುತ್ತೇನೆ"
    },
    "ಬತ್ತೀನಿ": {
        "ನಾನು":"ಬರುತ್ತೇನೆ","ನೀನು":"ಬರುತ್ತೀಯೆ",
        "ಅವನು":"ಬರುತ್ತಾನೆ","ಅವಳು":"ಬರುತ್ತಾಳೆ",
        "ಅವರು":"ಬರುತ್ತಾರೆ","ನಾವು":"ಬರುತ್ತೇವೆ",
        "DEFAULT":"ಬರುತ್ತೇನೆ"
    },
    "ಹೋಗ್ತಿವಿ": {
        "ನಾವು":"ಹೋಗುತ್ತೇವೆ","DEFAULT":"ಹೋಗುತ್ತೇವೆ"
    },
}

# ── Timeout wrapper ───────────────────────────────────────
def run_with_timeout(func, args=(), timeout_sec=15):
    result = [None]
    error  = [None]
    def target():
        try:
            result[0] = func(*args)
        except Exception as e:
            error[0] = e
    t = threading.Thread(target=target)
    t.daemon = True
    t.start()
    t.join(timeout_sec)
    if t.is_alive():
        return None, False
    if error[0]:
        return None, False
    return result[0], True

# ── Helper functions ──────────────────────────────────────
def get_differing_words(incorrect, correct):
    inc = incorrect.strip().split()
    cor = correct.strip().split()
    pairs = []
    for i, (iw, cw) in enumerate(zip(inc, cor)):
        ic = iw.strip('.,!?;:()[]{}"\'।')
        cc = cw.strip('.,!?;:()[]{}"\'।')
        if ic != cc and ic and cc:
            pairs.append((ic, cc, i, incorrect))
    return pairs

def check_in_suggestions(suggestions, correct_word, topk):
    return 1 if correct_word in suggestions[:topk] else 0

def safe_validate(word, full_sentence, position):
    def _v():
        return validator.validate_word(
            word, full_sentence=full_sentence, position=position
        )
    result, ok = run_with_timeout(_v, timeout_sec=15)
    if not ok:
        return {'valid': True, 'suggestions': [], 'confidence': 0.0}
    return result

def get_subject_from_sentence(sentence):
    words = sentence.split()
    for w in words:
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in MALE_PRONOUNS:
            return wc, 'MALE'
        if wc in FEMALE_PRONOUNS:
            return wc, 'FEMALE'
        if wc in PLURAL_PRONOUNS:
            return wc, 'PLURAL'
    return None, None

def get_gender_suggestions(word, sentence):
    suggestions = []
    clean = word.strip('.,!?;:()[]{}"\'।')

    if clean in IRREGULAR_VERB_MAP:
        stem, alt1, alt2 = IRREGULAR_VERB_MAP[clean]
        candidates = [alt1, alt2]
        return [s for s in candidates if s in validator.dictionary_words]

    current_gender = None
    suffix_used    = None
    for s in MALE_VERB_SUFFIXES:
        if clean.endswith(s):
            current_gender = 'MALE'
            suffix_used = s
            break
    if not current_gender:
        for s in FEMALE_VERB_SUFFIXES:
            if clean.endswith(s):
                current_gender = 'FEMALE'
                suffix_used = s
                break
    if not current_gender:
        for s in PLURAL_VERB_SUFFIXES:
            if clean.endswith(s):
                current_gender = 'PLURAL'
                suffix_used = s
                break

    if not current_gender or not suffix_used:
        return suggestions

    stem = clean[:-len(suffix_used)]
    all_suffixes = (
        [(s, 'MALE')   for s in MALE_VERB_SUFFIXES] +
        [(s, 'FEMALE') for s in FEMALE_VERB_SUFFIXES] +
        [(s, 'PLURAL') for s in PLURAL_VERB_SUFFIXES]
    )
    seen = set([clean])
    for s, gender in all_suffixes:
        candidate = stem + s
        if candidate not in seen:
            if candidate in validator.dictionary_words:
                suggestions.append(candidate)
                seen.add(candidate)

    return suggestions[:10]

def get_pronoun_suggestions(wrong_pronoun, entity_memory):
    suggestions = []
    for entity, gender in entity_memory.items():
        if gender == 'MALE' and wrong_pronoun in FEMALE_PRONOUNS:
            suggestions = ['ಅವನು', 'ಈತನು']
            break
        elif gender == 'FEMALE' and wrong_pronoun in MALE_PRONOUNS:
            suggestions = ['ಅವಳು', 'ಈಕೆ']
            break
        elif gender == 'MALE' and wrong_pronoun in PLURAL_PRONOUNS:
            suggestions = ['ಅವನು']
            break
    return suggestions

def detect_gender_mismatch(sentence):
    words = sentence.split()
    for i, w in enumerate(words):
        wc = w.strip('.,!?;:()[]{}"\'।')
        if wc in FEMALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return False
        elif wc in MALE_PRONOUNS:
            for fw in words[i+1:]:
                fwc = fw.strip('.,!?;:()[]{}"\'।')
                if any(fwc.endswith(s) for s in FEMALE_VERB_SUFFIXES):
                    return True
                if any(fwc.endswith(s) for s in MALE_VERB_SUFFIXES):
                    return False
    return False

def force_vibhakti_check(word, sentence):
    try:
        err, base, wrong, correct_marker = validator.vibhakti.detect_vibhakti_error(
            word, validator.dictionary_words
        )
        if err and base:
            corrected = validator.vibhakti.correct_vibhakti(word, base, correct_marker)
            if corrected and corrected != word:
                return [corrected]
    except Exception:
        pass
    return []

def force_postposition_check(word, position, sentence):
    try:
        tokens    = sentence.split()
        corrected = validator.postposition.correct(
            word, position, tokens, validator.dictionary_words
        )
        if corrected and corrected != word:
            return [corrected]
    except Exception:
        pass
    return []

def normalize_colloquial_sentence(incorrect, correct):
    inc_words = incorrect.split()

    colloquial_word = None
    for iw in inc_words:
        ic = iw.strip('.,!?;:()[]{}"\'।')
        if ic in COLLOQUIAL_VERB_MAP:
            colloquial_word = ic
            break

    if not colloquial_word:
        try:
            norm = colloquial.normalize_paragraph(incorrect)
            if norm.has_changes():
                return 1, norm.normalized_text.strip()
        except Exception:
            pass
        return 0, ''

    # Find subject pronoun first
    subject = None
    for iw in inc_words:
        ic = iw.strip('.,!?;:()[]{}"\'।')
        if ic in COLLOQUIAL_VERB_MAP[colloquial_word]:
            subject = ic
            break

    # If not found try any word in sentence against map keys
    if not subject:
        for iw in inc_words:
            ic = iw.strip('.,!?;:()[]{}"\'।')
            if ic in COLLOQUIAL_VERB_MAP[colloquial_word]:
                subject = ic
                break

    # Get standard form
    standard_form = ''
    if subject:
        standard_form = COLLOQUIAL_VERB_MAP[colloquial_word].get(subject, '')

    # Fallback to DEFAULT
    if not standard_form:
        standard_form = COLLOQUIAL_VERB_MAP[colloquial_word].get('DEFAULT', '')

    # Last fallback — first value in map
    if not standard_form:
        values = [v for k, v in COLLOQUIAL_VERB_MAP[colloquial_word].items()
                  if k != 'DEFAULT']
        standard_form = values[0] if values else ''

    if not standard_form:
        return 0, ''

    normalized = incorrect.replace(colloquial_word, standard_form, 1)
    return 1, normalized.strip()

# ── Per row evaluation ────────────────────────────────────
print('\nRunning evaluation...\n')
rows = []

for i, row in df.iterrows():
    incorrect  = str(row['Incorrect Sentence']).strip()
    correct    = str(row['Correct Sentence']).strip()
    error_type = str(row['Error Type']).strip()
    module     = str(row['Module']).strip()

    result_row = {
        'ID':        row['ID'],
        'Module':    module,
        'Error Type':error_type,
        'Incorrect': incorrect,
        'Correct':   correct,
        'Flagged':   0,
        'Top1':      0,
        'Top3':      0,
        'Top5':      0,
        'Top10':     0,
        'Detected':  0,
        'Notes':     ''
    }

    # ── Colloquial Normalizer ─────────────────────────────
    if module == 'Colloquial Normalizer':
        try:
            detected, normalized = normalize_colloquial_sentence(incorrect, correct)
            result_row['Detected'] = detected
            if detected:
                result_row['Flagged'] = 1
                if normalized == correct:
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = normalized
            else:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Honorific Agreement ───────────────────────────────
    if module == 'Honorific Agreement':
        try:
            h = honorific_process(incorrect)
            if h.correction_made:
                result_row['Detected'] = 1
                if h.corrected_sentence.strip() == correct.strip():
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                result_row['Notes'] = h.corrected_sentence
            else:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Context Memory Layer ──────────────────────────────
    if module == 'Context Memory Layer':
        try:
            ContextAnalyzer.build_entity_memory_from_paragraph(incorrect)
            memory = ContextAnalyzer.entity_memory.copy()
            if memory:
                result_row['Detected'] = 1

            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                if (inc_word in MALE_PRONOUNS or
                        inc_word in FEMALE_PRONOUNS or
                        inc_word in PLURAL_PRONOUNS):
                    sugg = get_pronoun_suggestions(inc_word, memory)
                    if not sugg:
                        if inc_word in FEMALE_PRONOUNS:
                            sugg = ['ಅವನು','ಈತನು','ಅವರು']
                        elif inc_word in MALE_PRONOUNS:
                            sugg = ['ಅವಳು','ಈಕೆ','ಅವರು']
                        else:
                            sugg = ['ಅವನು','ಅವಳು']
                else:
                    sugg = get_gender_suggestions(inc_word, sent)

                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Memory:{memory} Sugg:{sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── SOV Detection ─────────────────────────────────────
    if module == 'SOV Detection':
        try:
            words     = incorrect.split()
            sov_found = False
            for wi, word in enumerate(words):
                clean = word.strip('.,!?;:()[]{}"\'।')
                flag  = validator._check_sov_violation(clean, wi, incorrect)
                if flag:
                    sov_found = True
                    result_row['Detected'] = 1
                    result_row['Top1']  = 1
                    result_row['Top3']  = 1
                    result_row['Top5']  = 1
                    result_row['Top10'] = 1
                    result_row['Notes'] = flag
                    break
            if not sov_found:
                result_row['Notes'] = 'Not detected'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Gender Agreement ──────────────────────────────────
    if module == 'Gender Agreement':
        try:
            pairs    = get_differing_words(incorrect, correct)
            mismatch = detect_gender_mismatch(incorrect)
            if mismatch:
                result_row['Detected'] = 1

            for inc_word, cor_word, pos, sent in pairs:
                sugg = get_gender_suggestions(inc_word, sent)
                if not sugg:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])

                result_row['Flagged'] = 1 if sugg else 0
                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg:{sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── FST Morphology ────────────────────────────────────
    if module == 'FST Morphology':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                sugg = get_gender_suggestions(inc_word, sent)

                if not sugg or cor_word not in sugg:
                    fst_result = None
                    try:
                        fst_result = validator.morphology.correct_verb_form(
                            inc_word, sent, pos
                        )
                    except Exception:
                        pass
                    if fst_result and fst_result != inc_word:
                        if fst_result not in sugg:
                            sugg.insert(0, fst_result)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg:{sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Vibhakti Correction ───────────────────────────────
    if module == 'Vibhakti Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                sugg = force_vibhakti_check(inc_word, sent)
                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1
                else:
                    val  = safe_validate(inc_word, sent, pos)
                    sugg = val.get('suggestions', [])
                    if not val.get('valid', True):
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg:{sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Postposition Correction ───────────────────────────
    if module == 'Postposition Correction':
        try:
            inc_words = incorrect.split()
            cor_words = correct.split()
            found_fix = False

            for wi, (iw, cw) in enumerate(zip(inc_words, cor_words)):
                ic = iw.strip('.,!?;:()[]{}"\'।')
                cc = cw.strip('.,!?;:()[]{}"\'।')
                if ic != cc:
                    if cc.startswith(ic) and len(cc) > len(ic):
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                        result_row['Top1']  = 1
                        result_row['Top3']  = 1
                        result_row['Top5']  = 1
                        result_row['Top10'] = 1
                        result_row['Notes'] = f'Added postposition to {ic}'
                        found_fix = True
                        break
                    else:
                        sugg = force_postposition_check(ic, wi, incorrect)
                        if sugg:
                            result_row['Detected'] = 1
                            result_row['Flagged']  = 1
                            result_row['Top1']  = check_in_suggestions(sugg, cc, 1)
                            result_row['Top3']  = check_in_suggestions(sugg, cc, 3)
                            result_row['Top5']  = check_in_suggestions(sugg, cc, 5)
                            result_row['Top10'] = check_in_suggestions(sugg, cc, 10)
                            result_row['Notes'] = f'Sugg:{sugg[:3]}'
                            found_fix = True
                            break

            if not found_fix:
                for wi, iw in enumerate(inc_words):
                    ic = iw.strip('.,!?;:()[]{}"\'।')
                    sugg = force_postposition_check(ic, wi, incorrect)
                    if sugg:
                        result_row['Detected'] = 1
                        result_row['Flagged']  = 1
                        result_row['Notes'] = f'Sugg:{sugg[:3]}'
                        found_fix = True
                        break

            if not found_fix:
                result_row['Notes'] = 'Not detected'

        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Spelling Correction ───────────────────────────────
    if module == 'Spelling Correction':
        try:
            pairs = get_differing_words(incorrect, correct)
            for inc_word, cor_word, pos, sent in pairs:
                from new_validator import generate_ocr_candidates
                ocr_cands = [
                    c for c in generate_ocr_candidates(inc_word)
                    if c in validator.dictionary_words
                ]

                def _lev():
                    return validator._find_candidates_levenshtein(
                        inc_word, max_suggestions=10
                    )
                lev_sugg, ok = run_with_timeout(_lev, timeout_sec=15)
                if not ok or not lev_sugg:
                    lev_sugg = []

                sugg = ocr_cands.copy()
                seen = set(sugg)
                for s in lev_sugg:
                    if s not in seen:
                        sugg.append(s)
                        seen.add(s)

                if sugg:
                    result_row['Detected'] = 1
                    result_row['Flagged']  = 1

                result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
                result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
                result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
                result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
                result_row['Notes'] = f'Sugg:{sugg[:3]}'
        except Exception as e:
            result_row['Notes'] = f'Error: {e}'
        rows.append(result_row)
        continue

    # ── Extra Suffix Removal (default) ───────────────────
    try:
        pairs = get_differing_words(incorrect, correct)
        if not pairs:
            result_row['Notes'] = 'No differing words found'
            rows.append(result_row)
            continue

        for inc_word, cor_word, pos, sent in pairs:
            val      = safe_validate(inc_word, sent, pos)
            sugg     = val.get('suggestions', [])
            is_valid = val.get('valid', True)

            result_row['Flagged']  = 0 if is_valid else 1
            result_row['Detected'] = 0 if is_valid else 1
            result_row['Top1']  = check_in_suggestions(sugg, cor_word, 1)
            result_row['Top3']  = check_in_suggestions(sugg, cor_word, 3)
            result_row['Top5']  = check_in_suggestions(sugg, cor_word, 5)
            result_row['Top10'] = check_in_suggestions(sugg, cor_word, 10)
            result_row['Notes'] = f'Sugg:{sugg[:3]}'

    except Exception as e:
        result_row['Notes'] = f'Error: {e}'

    rows.append(result_row)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500] done')

# ── Build results ─────────────────────────────────────────
results_df = pd.DataFrame(rows)

# ── Module summary ────────────────────────────────────────
print('\n' + '='*70)
print('MODULE-WISE EVALUATION RESULTS')
print('='*70)

summary_rows = []
for mod, grp in results_df.groupby('Module'):
    total    = len(grp)
    detected = grp['Detected'].sum()
    top1     = grp['Top1'].sum()
    top3     = grp['Top3'].sum()
    top5     = grp['Top5'].sum()
    top10    = grp['Top10'].sum()

    print(f'\n{mod} ({total} cases)')
    print(f'  Detection Rate  : {detected/total*100:.1f}%')
    print(f'  Top-1 Accuracy  : {top1/total*100:.1f}%')
    print(f'  Top-3 Accuracy  : {top3/total*100:.1f}%')
    print(f'  Top-5 Accuracy  : {top5/total*100:.1f}%')
    print(f'  Top-10 Accuracy : {top10/total*100:.1f}%')

    summary_rows.append({
        'Module':              mod,
        'Total Cases':         total,
        'Detection Rate (%)':  round(detected/total*100, 1),
        'Top-1 Accuracy (%)':  round(top1/total*100, 1),
        'Top-3 Accuracy (%)':  round(top3/total*100, 1),
        'Top-5 Accuracy (%)':  round(top5/total*100, 1),
        'Top-10 Accuracy (%)': round(top10/total*100, 1),
    })

summary_df = pd.DataFrame(summary_rows)

total_all    = len(results_df)
detected_all = results_df['Detected'].sum()
top1_all     = results_df['Top1'].sum()
top3_all     = results_df['Top3'].sum()
top5_all     = results_df['Top5'].sum()

print('\n' + '='*70)
print('OVERALL ACROSS ALL 500 CASES')
print('='*70)
print(f'  Detection Rate : {detected_all/total_all*100:.1f}%')
print(f'  Top-1 Accuracy : {top1_all/total_all*100:.1f}%')
print(f'  Top-3 Accuracy : {top3_all/total_all*100:.1f}%')
print(f'  Top-5 Accuracy : {top5_all/total_all*100:.1f}%')

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='Full Results', index=False)
    summary_df.to_excel(writer, sheet_name='Module Summary', index=False)

print(f'\n✅ Results saved to {OUTPUT_PATH}')

[Validator] Loaded Padakosha_kannada_csv.csv
[Validator] Loaded combined_word_scrapped_csv.csv
[Validator] ✅ Total: 1,095,944 words
[EnhancedValidator] Initialized v7.1
  Dictionary: 1,095,944 words
  Context DB: ✅
  FST Morphology:        ✅
  Vibhakti Correction:   ✅
  Postposition Fix:      ✅
[ColloquialNormalizer]   Bengaluru_urban_colloquial.csv: 93 entries (Bangalore Urban Informal)
[ColloquialNormalizer]   costal_ktk_colloquial.csv: 97 entries (Coastal Karnataka)
[ColloquialNormalizer]   Malenadu_colloquial.csv: 81 entries (Malenadu)
[ColloquialNormalizer]   mysuru_colloquial.csv: 48 entries (Mysuru/Old Mysore)
[ColloquialNormalizer]   north_karnataka_colloquial.csv: 83 entries (North Karnataka)
[ColloquialNormalizer] Loaded 402 entries from 5 dialects
✅ All modules loaded
✅ Dataset: 500 rows

Running evaluation...

  [150/500] done
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMemory] 'ತಾತ' → MALE
[EntityMe

In [ ]:
* To create a public link, set `share=True` in `launch()`.
[LoadWords] Building word list and pre-computing suggestions...
[EntityMemory] 'ಪಾಠ' → MALE
[EntityMemory] Built: {'ಪಾಠ': 'MALE'}

In [1]:
import pandas as pd
df = pd.read_csv(r'F:\INTERFACE_GRADIO\data\500_excel.csv', encoding='utf-8')
print(df.columns.tolist())
print(df.shape)
df.head(3)

['ID', 'Incorrect Sentence', 'Correct Sentence', 'Error Type', 'Module']
(500, 5)


,ID,Incorrect Sentence,Correct Sentence,Error Type,Module
0,1,ನಾನು ನಿನ್ನೆ ಹೊಸ ಪುತ್ತಕ ಓದಿದೆ,ನಾನು ನಿನ್ನೆ ಹೊಸ ಪುಸ್ತಕ ಓದಿದೆ,Spelling Error,Spelling Correction
1,2,ನೀನು ನಿನ್ನೆ ಹೊಸ ಪುತ್ತಕ ಓದಿದೆ,ನೀನು ನಿನ್ನೆ ಹೊಸ ಪುಸ್ತಕ ಓದಿದೆ,Spelling Error,Spelling Correction
2,3,ಅವನು ನಿನ್ನೆ ಹೊಸ ಪುತ್ತಕ ಓದಿದನು,ಅವನು ನಿನ್ನೆ ಹೊಸ ಪುಸ್ತಕ ಓದಿದನು,Spelling Error,Spelling Correction
